# Emotional Tone and Fundraising Success
### Dictionary-based analysis of GoFundMe campaigns · GALC + EmoLex

**Project overview.** I studied how emotional language in campaign descriptions relates to fundraising outcomes. I used Python to prepare the dataset, SEANCE to measure dictionary-based emotion scores, and OLS regression with HC3 robust standard errors to examine relationships with the percentage of funding goals raised and number of donors. I compared the GALC and NRC EmoLex dictionaries and performed sensitivity analyses.

**Data and privacy.** The source file contains real campaign descriptions and identifying details. I have intentionally removed all saved notebook outputs and disabled raw-text previews. **Do not commit `Data_GoFundMe.csv`, the SEANCE input text files, unreviewed SEANCE exports, or other campaign-level records to a public repository without confirming consent, licensing, and privacy requirements.** This notebook asks for local input files during execution; they are not included in the public repository.

**Reproduction in Google Colab.** Run the code cells in order. When prompted, upload the source dataset `Data_GoFundMe.csv`, the original GALC SEANCE output `results.csv`, and the combined GALC + EmoLex SEANCE output `results_additional.csv`. Desktop SEANCE processing is a manual step described below. Required Python packages: `pandas`, `numpy`, `statsmodels`, `matplotlib`, `Pillow`. This notebook uses `google.colab.files`, so it is designed for Google Colab.

**Academic note.** The original GALC measures were explored before four constructs were selected. The additional EmoLex analysis was exploratory. Dictionary matches measure words in campaign text; they do not directly measure how donors felt, and regression associations do not establish causation.

In [ ]:
# STEP 1: Import and examine the GoFundMe dataset

# Import required libraries
import pandas as pd
import numpy as np
import os
from google.colab import files

# Upload the original dataset
uploaded = files.upload()

# Read the uploaded CSV file
filename = next(iter(uploaded))

df = pd.read_csv(
    filename,
    encoding="utf-8",
    encoding_errors="replace"
)

# Display dataset dimensions
print("Dataset shape:", df.shape)

# Display column names
print("\nColumn names:")
print(df.columns.tolist())

# Show only the shape and column names; do not print private campaign stories
print("\nDataset loaded. Campaign-level rows are intentionally hidden in public notebook outputs.")

## 1. Initial data review
I first inspected the raw fundraising data for missing values, duplicates, inconsistent number formats, and variables needed for the analysis.

In [ ]:
# STEP 2: DATA QUALITY ASSESSMENT

# 1. Check the dimensions
print("Total campaigns:", df.shape[0])
print("Total variables:", df.shape[1])

# 2. Check data types
print("\nDATA TYPES:")
print(df.dtypes)

# 3. Check missing values
print("\nMISSING VALUES:")
missing = df.isnull().sum()

missing_table = pd.DataFrame({
    "Missing Values": missing,
    "Missing Percentage": (missing / len(df) * 100).round(2)
})

display(missing_table)

# 4. Check duplicate records
print("\nDUPLICATE RECORDS:")
print("Exact duplicate rows:", df.duplicated().sum())

print("Duplicate Campaign IDs:",
      df["Unnamed: 0"].duplicated().sum())

# 5. Examine important variables
print("\nSAMPLE OF IMPORTANT VARIABLES:")

columns_to_check = [
    "Unnamed: 0",
    "Amount_Raised$",
    "Goal$",
    "Number_of_Donators",
    "Length_of_Fundraising",
    "FB_Shares",
    "GFM_hearts"
]

display(df[columns_to_check].head(10))

## 2. Data cleaning and outcomes
I converted amounts containing commas and suffixes such as K or M to numbers. I removed unusable records, converted durations to days (one month = 30 days), and calculated `Percent_Raised = 100 × Amount_Raised / Goal`. I excluded campaigns with $1–$2 nominal goals from the funding-percentage regressions because they create extreme percentages.

In [ ]:
# STEP 3A: CLEAN NUMERICAL VARIABLES

import pandas as pd
import numpy as np
import re

# Create a working copy of the original dataset
# Keep df unchanged so we can refer to the raw data later
df_clean = df.copy()

# Rename the campaign identifier
df_clean.rename(
    columns={"Unnamed: 0": "Campaign_ID"},
    inplace=True
)

# Function to convert values like:
# "1.0M" -> 1000000
# "26k"  -> 26000
# "12,167" -> 12167

def clean_number(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    # Remove currency symbols, commas, and spaces
    value = value.replace("$", "")
    value = value.replace(",", "")
    value = value.replace(" ", "")

    if value == "":
        return np.nan

    # Match numbers with optional K, M, or B suffix
    match = re.fullmatch(
        r"([+-]?(?:\d+\.?\d*|\.\d+))([KkMmBb]?)",
        value
    )

    if match is None:
        return np.nan

    number = float(match.group(1))
    suffix = match.group(2).lower()

    multipliers = {
        "": 1,
        "k": 1000,
        "m": 1000000,
        "b": 1000000000
    }

    return number * multipliers[suffix]


# Variables requiring numerical conversion
numeric_columns = [
    "Amount_Raised$",
    "Goal$",
    "Number_of_Donators",
    "FB_Shares",
    "GFM_hearts"
]

# Convert each variable
for col in numeric_columns:

    df_clean[col] = df_clean[col].apply(clean_number)

# Check results
print("DATA TYPES AFTER CLEANING:")
print(df_clean[numeric_columns].dtypes)

print("\nFIRST 10 CLEANED OBSERVATIONS:")

display(
    df_clean[
        ["Campaign_ID"] + numeric_columns
    ].head(10)
)

# Check how many values remain missing
print("\nMISSING VALUES AFTER CONVERSION:")

print(df_clean[numeric_columns].isna().sum())

# Identify any nonmissing raw values that could not be converted
print("\nVALUES THAT COULD NOT BE CONVERTED:")

for col in numeric_columns:

    failed = (
        df[col].notna()
        & df_clean[col].isna()
    )

    print(f"{col}: {failed.sum()} conversion failures")

    # Do not print the unconverted raw values: some fields may contain personal text.
    if failed.any():
        print("  Unconverted examples have been hidden for privacy.")

print("\nTOTAL CAMPAIGNS:", len(df_clean))

In [ ]:
# STEP 3B: IDENTIFY AND REMOVE UNUSABLE CAMPAIGNS

# Essential variables
required_columns = [
    "Amount_Raised$",
    "Goal$",
    "Number_of_Donators",
    "Length_of_Fundraising",
    "Text"
]

# Check whether essential information is present
complete_records = df_clean[required_columns].notna().all(axis=1)

# Check whether campaign descriptions contain text
valid_text = (
    df_clean["Text"].notna()
    & df_clean["Text"].astype(str).str.strip().ne("")
)

# Fundraising goals must be greater than zero
valid_goal = df_clean["Goal$"] > 0

# Identify usable observations
valid_records = complete_records & valid_text & valid_goal

# Display excluded campaigns
excluded = df_clean.loc[~valid_records].copy()

print("ORIGINAL CAMPAIGNS:", len(df_clean))
print("UNUSABLE CAMPAIGNS:", len(excluded))

print("\nCAMPAIGN IDs EXCLUDED:")
print(excluded["Campaign_ID"].tolist())

# Create cleaned working dataset
df_work = df_clean.loc[valid_records].copy()

# Reset row index
df_work.reset_index(drop=True, inplace=True)

print("\nCAMPAIGNS RETAINED:", len(df_work))

print("\nMISSING VALUES IN REQUIRED VARIABLES:")
print(df_work[required_columns].isna().sum())

print("\nFIRST FIVE CLEANED NUMERICAL OBSERVATIONS:")
display(df_work[["Campaign_ID", "Amount_Raised$", "Goal$", "Number_of_Donators", "FB_Shares", "GFM_hearts"]].head())

In [ ]:
# STEP 3C: CONVERT FUNDRAISING DURATION INTO DAYS

import pandas as pd
import numpy as np

# Extract the number and unit from fundraising duration
# Examples:
# "1 month"  -> 1, month
# "3 months" -> 3, months
# "12 days"  -> 12, days

duration = (
    df_work["Length_of_Fundraising"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.extract(r"^(\d+)\s*(days?|months?)$")
)

# Check for unexpected formats before conversion
if duration.isna().any(axis=1).any():
    print("Unrecognized duration values:")
    display(
        df_work.loc[
            duration.isna().any(axis=1),
            ["Campaign_ID", "Length_of_Fundraising"]
        ]
    )
    raise ValueError("Some fundraising durations could not be parsed.")

# Extract numerical value
duration_number = pd.to_numeric(duration[0])

# Convert months to 30 days
# Values already expressed in days remain unchanged
df_work["Length_Days"] = (
    duration_number *
    np.where(duration[1].str.startswith("month"), 30, 1)
).astype(int)

# Display conversion results
print("DURATION CONVERSION RESULTS:")

display(
    df_work[
        ["Campaign_ID", "Length_of_Fundraising", "Length_Days"]
    ].head(15)
)

# Display all distinct duration conversions
print("\nUNIQUE DURATION CONVERSIONS:")

display(
    df_work[
        ["Length_of_Fundraising", "Length_Days"]
    ]
    .drop_duplicates()
    .sort_values("Length_Days")
    .reset_index(drop=True)
)

# Verify the conversion
print("\nVERIFICATION:")

print("Total campaigns:", len(df_work))

print(
    "Missing duration values:",
    df_work["Length_Days"].isna().sum()
)

print(
    "Minimum duration:",
    df_work["Length_Days"].min(),
    "days"
)

print(
    "Maximum duration:",
    df_work["Length_Days"].max(),
    "days"
)

In [ ]:
# STEP 3D: CREATE FUNDRAISING SUCCESS VARIABLE

import pandas as pd
import numpy as np

# Calculate percentage of fundraising goal achieved
df_work["Percent_Raised"] = (
    df_work["Amount_Raised$"] /
    df_work["Goal$"]
) * 100

# Display first 10 campaigns
print("FIRST 10 CAMPAIGNS WITH FUNDING SUCCESS:")

display(
    df_work[
        [
            "Campaign_ID",
            "Amount_Raised$",
            "Goal$",
            "Percent_Raised"
        ]
    ].head(10).round(2)
)

# Summary statistics
print("\nDESCRIPTIVE STATISTICS OF FUNDING SUCCESS:")

print(
    df_work["Percent_Raised"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .round(2)
)

# Check different levels of fundraising success
print("\nFUNDRAISING SUCCESS CHECK:")

print(
    "Campaigns achieving 100% or more:",
    (df_work["Percent_Raised"] >= 100).sum()
)

print(
    "Campaigns achieving less than 100%:",
    (df_work["Percent_Raised"] < 100).sum()
)

print(
    "Missing Percent_Raised values:",
    df_work["Percent_Raised"].isna().sum()
)

print(
    "Negative Percent_Raised values:",
    (df_work["Percent_Raised"] < 0).sum()
)

print("\nTOP 5 HIGHEST FUNDRAISING PERCENTAGES:")

display(
    df_work[
        [
            "Campaign_ID",
            "Amount_Raised$",
            "Goal$",
            "Percent_Raised"
        ]
    ]
    .sort_values("Percent_Raised", ascending=False)
    .head(5)
    .round(2)
)

In [ ]:
# STEP 3E: INVESTIGATE EXTREME FUNDRAISING VALUES

# 1. Examine unusually small fundraising goals

print("CAMPAIGNS WITH SMALL FUNDRAISING GOALS:")

print("Goal exactly $1:",
      (df_work["Goal$"] == 1).sum())

print("Goal $10 or less:",
      (df_work["Goal$"] <= 10).sum())

print("Goal $100 or less:",
      (df_work["Goal$"] <= 100).sum())

print("Goal below $1,000:",
      (df_work["Goal$"] < 1000).sum())


# 2. Examine extreme fundraising percentages

print("\nEXTREME FUNDRAISING PERCENTAGES:")

print("Percent_Raised > 100%:",
      (df_work["Percent_Raised"] > 100).sum())

print("Percent_Raised > 500%:",
      (df_work["Percent_Raised"] > 500).sum())

print("Percent_Raised > 1000%:",
      (df_work["Percent_Raised"] > 1000).sum())

print("Percent_Raised > 10000%:",
      (df_work["Percent_Raised"] > 10000).sum())


# 3. Display campaigns with the highest percentages

print("\nTOP 15 EXTREME CAMPAIGNS:")

display(
    df_work[
        [
            "Campaign_ID",
            "Amount_Raised$",
            "Goal$",
            "Percent_Raised",
            "Number_of_Donators"
        ]
    ]
    .sort_values("Percent_Raised", ascending=False)
    .head(15)
    .round(2)
)


# 4. Compare mean and median

print("\nMEAN VS MEDIAN:")

print("Mean Percent_Raised:",
      round(df_work["Percent_Raised"].mean(), 2))

print("Median Percent_Raised:",
      round(df_work["Percent_Raised"].median(), 2))

In [ ]:
# STEP 3F: FLAG CAMPAIGNS WITH NOMINAL FUNDRAISING GOALS

# Identify campaigns with goals of $1 or $2
df_work["Nominal_Goal"] = df_work["Goal$"] <= 2

# Create a dataset eligible for Model A
df_modelA = df_work.loc[
    ~df_work["Nominal_Goal"]
].copy()

# Check sample sizes
print("ORIGINAL WORKING DATASET:", len(df_work))

print(
    "CAMPAIGNS WITH NOMINAL GOALS:",
    df_work["Nominal_Goal"].sum()
)

print(
    "CAMPAIGNS ELIGIBLE FOR MODEL A:",
    len(df_modelA)
)

print(
    "CAMPAIGNS RETAINED FOR SEANCE:",
    len(df_work)
)

# Compare fundraising percentage distributions
comparison = pd.DataFrame({
    "All Campaigns": df_work["Percent_Raised"].describe(),
    "Excluding Nominal Goals": df_modelA["Percent_Raised"].describe()
})

print("\nCOMPARISON OF FUNDRAISING SUCCESS:")

display(comparison.round(2))

# Verify that original data is preserved
print("\nFINAL VERIFICATION:")

print("df_work rows:", len(df_work))
print("df_modelA rows:", len(df_modelA))

## 3. Prepare campaign descriptions for SEANCE
I removed URLs and HTML markup, standardized whitespace, and counted words. I kept words and negations instead of applying heavy text filtering so the dictionaries could process the available emotional language.

In [ ]:
# STEP 4A: TEXT PREPROCESSING FOR SEANCE

import pandas as pd
import re
import html

# Function to clean campaign descriptions
def clean_text(text):

    # Handle missing text
    if pd.isna(text):
        return ""

    # Convert to string
    text = str(text)

    # Convert HTML entities into readable characters
    text = html.unescape(text)

    # Remove website URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Replace tabs, line breaks, and extra spaces
    text = re.sub(r"\s+", " ", text)

    # Remove leading and trailing spaces
    text = text.strip()

    return text


# Apply cleaning function to all campaign descriptions
df_work["Text_Clean"] = df_work["Text"].apply(clean_text)


# Calculate number of words in cleaned descriptions
df_work["Word_Count"] = (
    df_work["Text_Clean"]
    .str.split()
    .str.len()
)


# Check total campaigns
print("TOTAL CAMPAIGNS:", len(df_work))


# Check empty descriptions after cleaning
print(
    "EMPTY DESCRIPTIONS AFTER CLEANING:",
    (df_work["Text_Clean"] == "").sum()
)


# Examine word count statistics
print("\nWORD COUNT STATISTICS:")

print(
    df_work["Word_Count"]
    .describe()
    .round(2)
)


# Privacy note: raw campaign descriptions are not printed in this public notebook.
# Check the resulting length distribution without revealing campaign stories.
print("\nTEXT PREPROCESSING COMPLETE. Original and cleaned descriptions are kept in memory only.")
print("Cleaned descriptions:", df_work["Text_Clean"].notna().sum())


### Export SEANCE input files
The next cell writes **one `.txt` file per campaign**, using the campaign ID as the filename. The resulting ZIP contains source descriptions and should be treated as private research material. It is generated locally for analysis, **not for public GitHub upload**.

In [ ]:
# STEP 4B: CREATE INDIVIDUAL TEXT FILES FOR SEANCE

import os
import shutil
from google.colab import files

# Step 1: Create a folder for SEANCE input files
folder_name = "GoFundMe_SEANCE_input"

# Remove an older copy of this generated folder, if it exists
if os.path.exists(folder_name):
    shutil.rmtree(folder_name)

os.makedirs(folder_name)


# Step 2: Verify that Campaign_ID values are unique
assert df_work["Campaign_ID"].is_unique, \
    "Campaign_ID values must be unique."


# Step 3: Create one text file for each campaign

for index, row in df_work.iterrows():

    # Use Campaign_ID as filename
    campaign_id = str(int(row["Campaign_ID"]))

    filename = f"{campaign_id}.txt"

    # Full file path
    filepath = os.path.join(folder_name, filename)

    # Extract cleaned description
    text = row["Text_Clean"]

    # Save text file
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(text)


# Step 4: Verify number of files created

txt_files = [
    f for f in os.listdir(folder_name)
    if f.endswith(".txt")
]

print("TOTAL CAMPAIGNS:", len(df_work))

print("TOTAL TEXT FILES CREATED:", len(txt_files))

print(
    "FILES MATCH CAMPAIGNS:",
    len(txt_files) == len(df_work)
)


# Step 5: Check created filenames only (without displaying campaign text).
print("\nEXAMPLE TEXT FILE NAMES (NO CONTENT DISPLAYED):")
print(sorted(txt_files)[:3])


# Step 6: Compress the folder into a ZIP file

zip_filename = shutil.make_archive(
    "GoFundMe_SEANCE_input",
    "zip",
    root_dir=".",
    base_dir=folder_name
)

print("\nZIP FILE CREATED:", zip_filename)

print(
    "ZIP FILE SIZE (MB):",
    round(os.path.getsize(zip_filename) / (1024 * 1024), 2)
)


# Step 7: Download ZIP file to your computer

files.download(zip_filename)

## 4. SEANCE dictionary scoring (manual desktop step)
SEANCE is a separate desktop application, so I ran it outside Colab with the exported text folder. For the **original run**, I selected GALC, All Words, and no component analysis. The stored original export (`results.csv`) also contains Lasswell indices, which were not used in the models. For the **additional run**, I selected both GALC and EmoLex and used the same 1,231 text files; the resulting export was saved as `results_additional.csv`. I used the non-negation-control setting in these runs.

The next cell imports the original SEANCE CSV and joins scores to campaign outcomes by campaign ID. Do not upload SEANCE text or raw campaign-level exports publicly until you have confirmed that they are appropriate to share.

In [ ]:
# STEP 6: MERGE SEANCE RESULTS WITH GOFUNDME DATA

import pandas as pd
from google.colab import files

# 1. Upload SEANCE output
print("Please upload your SEANCE results.csv file")

uploaded = files.upload()

filename = next(iter(uploaded))

# 2. Read SEANCE results
seance = pd.read_csv(filename, encoding="utf-8-sig")

print("\nSEANCE DATASET SHAPE:")
print(seance.shape)


# 3. Extract Campaign_ID from filename
# Example: "300.txt" -> 300

seance["Campaign_ID"] = (
    seance["filename"]
    .astype(str)
    .str.strip()
    .str.replace("\\", "/", regex=False)
    .str.split("/")
    .str[-1]
    .str.extract(r"^(\d+)\.txt$", expand=False)
)

# Check filenames before converting IDs
assert seance["Campaign_ID"].notna().all(), \
    "Some filenames do not contain valid Campaign IDs."

seance["Campaign_ID"] = seance["Campaign_ID"].astype(int)


# 4. Check for duplicate Campaign IDs
print("\nDUPLICATE CAMPAIGN IDs IN SEANCE:")
print(seance["Campaign_ID"].duplicated().sum())


# 5. Merge both datasets using Campaign_ID
df_merged = df_work.merge(
    seance,
    on="Campaign_ID",
    how="left",
    validate="one_to_one",
    indicator=True
)


# 6. Verify merging
print("\nMERGE RESULTS:")

print("Original campaigns:", len(df_work))
print("SEANCE records:", len(seance))
print("Merged campaigns:", len(df_merged))

print("\nMERGE STATUS:")
print(df_merged["_merge"].value_counts())


# 7. Check missing SEANCE results
print("\nCAMPAIGNS WITHOUT SEANCE SCORES:")
print(df_merged["nwords"].isna().sum())


# 8. Display merged dataset
print("\nEXAMPLE MERGED OBSERVATIONS:")

display(
    df_merged[
        [
            "Campaign_ID",
            "Percent_Raised",
            "Number_of_Donators",
            "Hope_GALC",
            "Compassion_GALC",
            "Fear_GALC",
            "Gratitude_GALC"
        ]
    ].head(10)
)


# 9. Final verification
assert len(df_merged) == len(df_work), \
    "The number of campaigns changed during merging."

assert (df_merged["_merge"] == "both").all(), \
    "Some campaigns did not match SEANCE results."

print("\nSUCCESS! ALL CAMPAIGNS MERGED CORRECTLY.")

## 5. Explore GALC emotions
After scoring, I reviewed the prevalence and variation of GALC constructs. I selected Feeling Love, Gratitude, Hope, and Anger because of their relevance to emotional connection, appreciation, future hope, and urgency. The selection followed inspection of the GALC output; it was not preregistered.

In [ ]:
# STEP 7A: EXAMINE SEANCE EMOTIONAL CONSTRUCTS

import pandas as pd
import numpy as np

# 1. Identify all GALC emotional variables
galc_columns = [
    col for col in df_merged.columns
    if col.endswith("_GALC")
]

print("TOTAL GALC CONSTRUCTS:", len(galc_columns))


# 2. Extract GALC scores as numerical values
galc_data = df_merged[galc_columns].apply(
    pd.to_numeric,
    errors="coerce"
)


# 3. Calculate descriptive statistics
emotion_summary = pd.DataFrame({

    "Nonzero_Campaigns": (galc_data > 0).sum(),

    "Nonzero_Percent": (
        (galc_data > 0).sum() / len(df_merged) * 100
    ),

    "Mean": galc_data.mean(),

    "Std_Dev": galc_data.std(),

    "Maximum": galc_data.max(),

    "Unique_Values": galc_data.nunique()

})


# 4. Sort emotions by number of nonzero observations
emotion_summary = emotion_summary.sort_values(
    "Nonzero_Campaigns",
    ascending=False
)


# 5. Display summary table
print("\nDISTRIBUTION OF ALL GALC EMOTIONAL CONSTRUCTS:")

display(emotion_summary.round(4))


# 6. Identify constructs with no variation
print("\nCONSTRUCTS WITH ZERO VARIATION:")

zero_variation = emotion_summary[
    emotion_summary["Std_Dev"] == 0
]

print(zero_variation.index.tolist())


# 7. Examine our initially proposed constructs
print("\nINITIAL CANDIDATE CONSTRUCTS:")

initial_constructs = [
    "Hope_GALC",
    "Compassion_GALC",
    "Fear_GALC",
    "Gratitude_GALC"
]

display(
    emotion_summary.loc[initial_constructs].round(4)
)


# 8. Identify constructs found in at least 5% of campaigns
print("\nCONSTRUCTS DETECTED IN AT LEAST 5% OF CAMPAIGNS:")

frequent_emotions = emotion_summary[
    emotion_summary["Nonzero_Percent"] >= 5
]

display(frequent_emotions.round(4))

In [ ]:
# STEP 7B: SELECT EMOTIONAL CONSTRUCTS AND CHECK CORRELATIONS

import pandas as pd
import numpy as np

# 1. Select four theoretically relevant emotional constructs

selected_emotions = [
    "Feelinglove_GALC",
    "Gratitude_GALC",
    "Hope_GALC",
    "Anger_GALC"
]

print("SELECTED EMOTIONAL CONSTRUCTS:")
print(selected_emotions)


# 2. Check descriptive statistics for selected emotions

print("\nDESCRIPTIVE STATISTICS:")

display(
    df_merged[selected_emotions]
    .describe()
    .round(4)
)


# 3. Check how many campaigns contain each emotion

print("\nEMOTIONAL CONSTRUCT DETECTION:")

detection = pd.DataFrame({
    "Nonzero_Campaigns":
        (df_merged[selected_emotions] > 0).sum(),

    "Nonzero_Percent":
        (df_merged[selected_emotions] > 0).mean() * 100
})

display(detection.round(2))


# 4. Calculate Pearson correlation matrix

print("\nCORRELATION MATRIX:")

correlation_matrix = (
    df_merged[selected_emotions]
    .corr()
)

display(correlation_matrix.round(3))


# 5. Check strong correlations between selected emotions

print("\nSTRONG CORRELATION CHECK:")

strong_pairs = []

for i in range(len(selected_emotions)):

    for j in range(i + 1, len(selected_emotions)):

        var1 = selected_emotions[i]
        var2 = selected_emotions[j]

        correlation = correlation_matrix.loc[var1, var2]

        if abs(correlation) >= 0.70:
            strong_pairs.append(
                (var1, var2, round(correlation, 3))
            )

if strong_pairs:
    print("Pairs with absolute correlation >= 0.70:")
    for pair in strong_pairs:
        print(pair)
else:
    print("No emotional construct pairs have correlation >= 0.70.")


# 6. Identify complete observations for future regression models

predictors = selected_emotions + [
    "GFM_hearts",
    "FB_Shares",
    "Length_Days"
]

# Model A excludes nominal-goal campaigns
modelA_ready = df_merged.loc[
    ~df_merged["Nominal_Goal"]
].dropna(
    subset=["Percent_Raised"] + predictors
)

# Model B retains nominal-goal campaigns
modelB_ready = df_merged.dropna(
    subset=["Number_of_Donators"] + predictors
)

print("\nPRELIMINARY REGRESSION SAMPLE SIZES:")

print("Model A observations:", len(modelA_ready))
print("Model B observations:", len(modelB_ready))

print("\nSTEP 7B COMPLETED.")

## 6. GALC regression analyses
I estimated two models using four GALC emotions, GoFundMe hearts, Facebook shares, and fundraising duration. Model A predicts percentage raised; Model B predicts number of donors. Both use OLS with HC3 heteroskedasticity-robust standard errors. I checked predictor collinearity with VIF.

In [ ]:
# STEP 8: CHECK MULTICOLLINEARITY USING VIF

import pandas as pd
import numpy as np
import statsmodels.api as sm

from statsmodels.stats.outliers_influence import variance_inflation_factor


# 1. Define all independent variables

predictors = [
    "Feelinglove_GALC",
    "Gratitude_GALC",
    "Hope_GALC",
    "Anger_GALC",
    "GFM_hearts",
    "FB_Shares",
    "Length_Days"
]


# 2. Prepare Model A dataset

df_A_reg = df_merged.loc[
    ~df_merged["Nominal_Goal"]
].dropna(
    subset=["Percent_Raised"] + predictors
).copy()


# 3. Prepare Model B dataset

df_B_reg = df_merged.dropna(
    subset=["Number_of_Donators"] + predictors
).copy()


# 4. Create a function to calculate VIF

def calculate_vif(data, predictor_names):

    X = data[predictor_names].astype(float)

    # Add intercept
    X = sm.add_constant(X, has_constant="add")

    vif_results = pd.DataFrame({
        "Variable": predictor_names,

        "VIF": [
            variance_inflation_factor(
                X.values,
                X.columns.get_loc(variable)
            )
            for variable in predictor_names
        ]
    })

    return vif_results


# 5. Calculate VIF for Model A

print("MODEL A: MULTICOLLINEARITY CHECK")

vif_A = calculate_vif(df_A_reg, predictors)

display(vif_A.round(3))


# 6. Calculate VIF for Model B

print("\nMODEL B: MULTICOLLINEARITY CHECK")

vif_B = calculate_vif(df_B_reg, predictors)

display(vif_B.round(3))


# 7. Identify potentially concerning VIF values

print("\nMODEL A: VARIABLES WITH VIF >= 5")

print(vif_A.loc[vif_A["VIF"] >= 5, "Variable"].tolist())


print("\nMODEL B: VARIABLES WITH VIF >= 5")

print(vif_B.loc[vif_B["VIF"] >= 5, "Variable"].tolist())


# 8. Final sample sizes

print("\nFINAL REGRESSION SAMPLE SIZES:")

print("Model A:", len(df_A_reg))

print("Model B:", len(df_B_reg))

print("\nSTEP 8 COMPLETED.")

In [ ]:
# STEP 9: REGRESSION MODEL A
# Dependent Variable: Percent_Raised

import pandas as pd
import numpy as np
import statsmodels.api as sm

# 1. Define independent variables

predictors = [
    "Feelinglove_GALC",
    "Gratitude_GALC",
    "Hope_GALC",
    "Anger_GALC",
    "GFM_hearts",
    "FB_Shares",
    "Length_Days"
]


# 2. Prepare dependent and independent variables

X_A = df_A_reg[predictors].astype(float)

y_A = df_A_reg["Percent_Raised"].astype(float)


# 3. Add constant (intercept) to regression

X_A = sm.add_constant(X_A)


# 4. Estimate OLS regression
# HC3 = heteroskedasticity-robust standard errors

model_A = sm.OLS(y_A, X_A).fit(
    cov_type="HC3"
)


# 5. Display complete regression summary

print("=" * 70)
print("MODEL A: PREDICTING FUNDRAISING SUCCESS")
print("=" * 70)

print(model_A.summary())


# 6. Create a clean regression results table

results_A = pd.DataFrame({

    "Variable": model_A.params.index,

    "Coefficient": model_A.params.values,

    "Robust_SE": model_A.bse.values,

    "P_Value": model_A.pvalues.values,

    "CI_Lower": model_A.conf_int().iloc[:, 0].values,

    "CI_Upper": model_A.conf_int().iloc[:, 1].values

})


print("\nMODEL A: REGRESSION RESULTS TABLE")

display(
    results_A.round(4)
)


# 7. Display model performance

print("\nMODEL A PERFORMANCE:")

print("Observations:", int(model_A.nobs))

print("R-squared:", round(model_A.rsquared, 4))

print("Adjusted R-squared:", round(model_A.rsquared_adj, 4))


# 8. Identify statistically significant predictors

print("\nSTATISTICALLY SIGNIFICANT PREDICTORS (p < 0.05):")

significant_A = results_A[
    (results_A["P_Value"] < 0.05) &
    (results_A["Variable"] != "const")
]

if len(significant_A) > 0:

    display(significant_A.round(4))

else:

    print("No predictors are significant at p < 0.05.")


print("\nSTEP 9 COMPLETED.")

### Sensitivity: extreme funding percentages
Some small funding goals create extreme percentages. I repeated Model A after excluding observations above the 99th percentile of `Percent_Raised` in the regression sample. This is a robustness check; the primary model remains unchanged.

In [ ]:
# STEP 9B: SENSITIVITY ANALYSIS FOR MODEL A

import pandas as pd
import numpy as np
import statsmodels.api as sm

# 1. Calculate the 99th percentile of Percent_Raised

cutoff_99 = df_A_reg["Percent_Raised"].quantile(0.99)

print("99TH PERCENTILE CUTOFF:")
print(round(cutoff_99, 2))


# 2. Identify observations above the 99th percentile

extreme_campaigns = df_A_reg[
    df_A_reg["Percent_Raised"] > cutoff_99
].copy()

print("\nNUMBER OF EXTREME CAMPAIGNS:")
print(len(extreme_campaigns))


# 3. Display extreme campaigns

print("\nEXTREME CAMPAIGNS:")

display(
    extreme_campaigns[
        [
            "Campaign_ID",
            "Amount_Raised$",
            "Goal$",
            "Percent_Raised"
        ]
    ]
    .sort_values("Percent_Raised", ascending=False)
    .round(2)
)


# 4. Create a separate sensitivity dataset

df_A_sensitivity = df_A_reg[
    df_A_reg["Percent_Raised"] <= cutoff_99
].copy()


# 5. Prepare regression variables

X_sensitivity = sm.add_constant(
    df_A_sensitivity[predictors].astype(float),
    has_constant="add"
)

y_sensitivity = df_A_sensitivity[
    "Percent_Raised"
].astype(float)


# 6. Run the sensitivity regression using HC3 robust SE

model_A_sensitivity = sm.OLS(
    y_sensitivity,
    X_sensitivity
).fit(cov_type="HC3")


# 7. Compare original and sensitivity models

comparison_A = pd.DataFrame({

    "Variable": model_A.params.index,

    "Original_Coefficient": model_A.params.values,

    "Original_P": model_A.pvalues.values,

    "Sensitivity_Coefficient":
        model_A_sensitivity.params.values,

    "Sensitivity_P":
        model_A_sensitivity.pvalues.values

})


print("\nREGRESSION COMPARISON:")

display(comparison_A.round(4))


# 8. Compare model performance

performance_A = pd.DataFrame({

    "Measure": [
        "Observations",
        "R-squared",
        "Adjusted R-squared"
    ],

    "Original_Model": [
        model_A.nobs,
        model_A.rsquared,
        model_A.rsquared_adj
    ],

    "Sensitivity_Model": [
        model_A_sensitivity.nobs,
        model_A_sensitivity.rsquared,
        model_A_sensitivity.rsquared_adj
    ]

})


print("\nMODEL PERFORMANCE COMPARISON:")

display(performance_A.round(4))


print("\nSTEP 9B COMPLETED.")

In [ ]:
# STEP 10: REGRESSION MODEL B
# Dependent Variable: Number_of_Donators

import pandas as pd
import numpy as np
import statsmodels.api as sm


# 1. Define independent variables

predictors = [
    "Feelinglove_GALC",
    "Gratitude_GALC",
    "Hope_GALC",
    "Anger_GALC",
    "GFM_hearts",
    "FB_Shares",
    "Length_Days"
]


# 2. Prepare Model B dataset

X_B = df_B_reg[predictors].astype(float)

y_B = df_B_reg["Number_of_Donators"].astype(float)


# 3. Add regression intercept

X_B = sm.add_constant(
    X_B,
    has_constant="add"
)


# 4. Estimate OLS regression with HC3 robust SE

model_B = sm.OLS(
    y_B,
    X_B
).fit(cov_type="HC3")


# 5. Display complete regression summary

print("=" * 70)
print("MODEL B: PREDICTING NUMBER OF DONORS")
print("=" * 70)

print(model_B.summary())


# 6. Create regression results table

results_B = pd.DataFrame({

    "Variable": model_B.params.index,

    "Coefficient": model_B.params.values,

    "Robust_SE": model_B.bse.values,

    "P_Value": model_B.pvalues.values,

    "CI_Lower": model_B.conf_int().iloc[:, 0].values,

    "CI_Upper": model_B.conf_int().iloc[:, 1].values

})


print("\nMODEL B: REGRESSION RESULTS TABLE")

display(results_B.round(4))


# 7. Display model performance

print("\nMODEL B PERFORMANCE:")

print("Observations:", int(model_B.nobs))

print("R-squared:", round(model_B.rsquared, 4))

print(
    "Adjusted R-squared:",
    round(model_B.rsquared_adj, 4)
)

print(
    "Overall model p-value:",
    round(model_B.f_pvalue, 6)
)


# 8. Identify statistically significant predictors

print("\nSTATISTICALLY SIGNIFICANT PREDICTORS (p < 0.05):")

significant_B = results_B[
    (results_B["P_Value"] < 0.05) &
    (results_B["Variable"] != "const")
]

if len(significant_B) > 0:

    display(significant_B.round(4))

else:

    print("No predictors are significant at p < 0.05.")


# 9. Examine distribution of donor counts

print("\nDONOR COUNT DESCRIPTIVE STATISTICS:")

print(
    y_B.describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    ).round(2)
)


print("\nSTEP 10 COMPLETED.")

### Diagnostic: overlap between GoFundMe hearts and donors
The hearts measure is very highly correlated with donor counts in the source data. Therefore, the high R² in donor models must be interpreted cautiously. I also re-estimated Model B without the hearts predictor.

In [ ]:
# STEP 10B: INVESTIGATE DONOR COUNT AND GFM HEARTS

import pandas as pd
import numpy as np

# 1. Create a separate diagnostic dataset
diagnostic = df_B_reg[
    [
        "Campaign_ID",
        "Number_of_Donators",
        "GFM_hearts"
    ]
].copy()


# 2. Calculate Pearson correlation

correlation = diagnostic[
    ["Number_of_Donators", "GFM_hearts"]
].corr()

print("CORRELATION BETWEEN DONORS AND GFM HEARTS:")

display(correlation.round(4))


# 3. Calculate differences between donors and hearts

diagnostic["Difference"] = (
    diagnostic["GFM_hearts"]
    - diagnostic["Number_of_Donators"]
)

diagnostic["Absolute_Difference"] = (
    diagnostic["Difference"].abs()
)


# 4. Display descriptive statistics

print("\nDESCRIPTIVE STATISTICS:")

display(
    diagnostic[
        [
            "Number_of_Donators",
            "GFM_hearts",
            "Difference",
            "Absolute_Difference"
        ]
    ].describe().round(2)
)


# 5. Check how often the values are close

print("\nSIMILARITY CHECK:")

print(
    "Exactly equal:",
    (diagnostic["Difference"] == 0).sum()
)

print(
    "Difference within 5:",
    (diagnostic["Absolute_Difference"] <= 5).sum()
)

print(
    "Difference within 10:",
    (diagnostic["Absolute_Difference"] <= 10).sum()
)

print(
    "Difference within 50:",
    (diagnostic["Absolute_Difference"] <= 50).sum()
)

print(
    "Median absolute difference:",
    diagnostic["Absolute_Difference"].median()
)


# 6. Display the first 15 campaigns

print("\nFIRST 15 CAMPAIGNS:")

display(
    diagnostic.head(15)
)


# 7. Display campaigns with the largest differences

print("\nTOP 10 CAMPAIGNS WITH LARGEST DIFFERENCES:")

display(
    diagnostic
    .sort_values(
        "Absolute_Difference",
        ascending=False
    )
    .head(10)
)


print("\nSTEP 10B COMPLETED.")

In [ ]:
# STEP 10C: MODEL B SENSITIVITY ANALYSIS
# Remove GFM_hearts to investigate measurement overlap

import pandas as pd
import numpy as np
import statsmodels.api as sm


# 1. Define predictors without GFM_hearts

predictors_no_hearts = [
    "Feelinglove_GALC",
    "Gratitude_GALC",
    "Hope_GALC",
    "Anger_GALC",
    "FB_Shares",
    "Length_Days"
]


# 2. Prepare dependent and independent variables

X_B_no_hearts = df_B_reg[
    predictors_no_hearts
].astype(float)

y_B_no_hearts = df_B_reg[
    "Number_of_Donators"
].astype(float)


# 3. Add intercept

X_B_no_hearts = sm.add_constant(
    X_B_no_hearts,
    has_constant="add"
)


# 4. Estimate OLS using HC3 robust standard errors

model_B_no_hearts = sm.OLS(
    y_B_no_hearts,
    X_B_no_hearts
).fit(cov_type="HC3")


# 5. Display full regression summary

print("=" * 70)
print("MODEL B: WITHOUT GOFUNDME HEARTS")
print("=" * 70)

print(model_B_no_hearts.summary())


# 6. Create clean regression results table

results_B_no_hearts = pd.DataFrame({

    "Variable": model_B_no_hearts.params.index,

    "Coefficient": model_B_no_hearts.params.values,

    "Robust_SE": model_B_no_hearts.bse.values,

    "P_Value": model_B_no_hearts.pvalues.values,

    "CI_Lower":
        model_B_no_hearts.conf_int().iloc[:, 0].values,

    "CI_Upper":
        model_B_no_hearts.conf_int().iloc[:, 1].values

})


print("\nREGRESSION RESULTS WITHOUT HEARTS:")

display(results_B_no_hearts.round(4))


# 7. Compare original and sensitivity models

comparison_B = pd.DataFrame({

    "Variable": model_B_no_hearts.params.index,

    "Original_Coefficient":
        model_B.params.reindex(
            model_B_no_hearts.params.index
        ).values,

    "Original_P":
        model_B.pvalues.reindex(
            model_B_no_hearts.params.index
        ).values,

    "Without_Hearts_Coefficient":
        model_B_no_hearts.params.values,

    "Without_Hearts_P":
        model_B_no_hearts.pvalues.values

})


print("\nCOEFFICIENT COMPARISON:")

display(comparison_B.round(4))


# 8. Compare model performance

performance_B = pd.DataFrame({

    "Measure": [
        "Observations",
        "R-squared",
        "Adjusted R-squared"
    ],

    "Original_Model": [
        model_B.nobs,
        model_B.rsquared,
        model_B.rsquared_adj
    ],

    "Without_Hearts": [
        model_B_no_hearts.nobs,
        model_B_no_hearts.rsquared,
        model_B_no_hearts.rsquared_adj
    ]

})


print("\nMODEL PERFORMANCE COMPARISON:")

display(performance_B.round(4))


# 9. Identify significant predictors without hearts

print("\nSIGNIFICANT PREDICTORS WITHOUT HEARTS (p < 0.05):")

significant_no_hearts = results_B_no_hearts[
    (results_B_no_hearts["P_Value"] < 0.05) &
    (results_B_no_hearts["Variable"] != "const")
]

if len(significant_no_hearts) > 0:

    display(significant_no_hearts.round(4))

else:

    print("No statistically significant predictors.")


print("\nSTEP 10C COMPLETED.")

## 7. Summarize GALC regression results
The following cells organize statistical results into tables. Table images are optional presentation outputs and are not necessary for running the regressions.

In [ ]:
# STEP 11: CREATE FINAL REGRESSION RESULTS TABLE

import pandas as pd
import numpy as np

# 1. Define variable names for the final report

variable_names = {
    "const": "Intercept",
    "Feelinglove_GALC": "Feeling Love",
    "Gratitude_GALC": "Gratitude",
    "Hope_GALC": "Hope",
    "Anger_GALC": "Anger",
    "GFM_hearts": "GoFundMe Hearts",
    "FB_Shares": "Facebook Shares",
    "Length_Days": "Fundraising Duration"
}


# 2. Create a function to format regression results

def format_regression_results(model):

    table = pd.DataFrame({
        "Coefficient": model.params,
        "Robust SE": model.bse,
        "P-value": model.pvalues
    })

    return table


# 3. Extract Model A results

table_A = format_regression_results(model_A)

table_A.columns = [
    "Model A Coefficient",
    "Model A Robust SE",
    "Model A P-value"
]


# 4. Extract Model B results

table_B = format_regression_results(model_B)

table_B.columns = [
    "Model B Coefficient",
    "Model B Robust SE",
    "Model B P-value"
]


# 5. Combine both regression models

final_table = table_A.join(table_B)


# 6. Replace technical variable names

final_table.index = final_table.index.map(
    variable_names
)

final_table.index.name = "Independent Variable"


# 7. Format p-values

def format_pvalue(p):

    if p < 0.001:
        return "<0.001"

    return f"{p:.3f}"


for column in [
    "Model A P-value",
    "Model B P-value"
]:

    final_table[column] = (
        final_table[column]
        .apply(format_pvalue)
    )


# 8. Round numerical coefficients and standard errors

numeric_columns = [
    "Model A Coefficient",
    "Model A Robust SE",
    "Model B Coefficient",
    "Model B Robust SE"
]

final_table[numeric_columns] = (
    final_table[numeric_columns].round(3)
)


# 9. Display final regression table

print("FINAL REGRESSION RESULTS TABLE")

display(final_table)


# 10. Create model performance table

model_summary = pd.DataFrame({

    "Statistic": [
        "Observations",
        "R-squared",
        "Adjusted R-squared"
    ],

    "Model A": [
        int(model_A.nobs),
        round(model_A.rsquared, 4),
        round(model_A.rsquared_adj, 4)
    ],

    "Model B": [
        int(model_B.nobs),
        round(model_B.rsquared, 4),
        round(model_B.rsquared_adj, 4)
    ]

})

print("\nMODEL PERFORMANCE SUMMARY")

display(model_summary)


print("\nSTEP 11 COMPLETED.")

### Optional: build styled presentation tables
This cell creates high-resolution tables using Pillow and downloads an archive in Colab. It does not change any fitted regression models. If only analysis code is needed, this step can be skipped.

In [ ]:
# ================================================================
# IS 705 — HOMEWORK 3
# PROFESSIONAL ACADEMIC REPORT TABLE GENERATOR
# ================================================================
#
# Creates:
#
# TABLE 1: Dataset Overview and Preprocessing
# TABLE 2: SEANCE Emotional Construct Descriptive Statistics
# TABLE 3: Main Regression Results (Model A and Model B)
# TABLE 4: Regression Sensitivity Analysis
#
# OUTPUT:
# - Four high-resolution PNG tables (300 DPI)
# - Four editable CSV tables
# - One ZIP file containing everything
#
# IMPORTANT:
# Run this in your EXISTING Google Colab notebook.
# Previous analysis variables must be available.
# ================================================================


# ================================================================
# PART 1 — IMPORT LIBRARIES
# ================================================================

import os
import zipfile
import shutil
import numpy as np
import pandas as pd

from PIL import Image, ImageDraw, ImageFont
from matplotlib import font_manager
from google.colab import files


# ================================================================
# PART 2 — VERIFY PREVIOUS ANALYSIS RESULTS
# ================================================================

required_objects = [
    "df",
    "df_work",
    "df_merged",
    "model_A",
    "model_B",
    "model_A_sensitivity",
    "model_B_no_hearts"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "Missing previous analysis objects: "
        + ", ".join(missing_objects)
        + "\nPlease rerun the earlier analysis cells."
    )

print("All required analysis results are available.")


# ================================================================
# PART 3 — CREATE OUTPUT FOLDER
# ================================================================

output_dir = "/content/IS705_HW3_Report_Tables"

os.makedirs(output_dir, exist_ok=True)

print("Output folder created.")


# ================================================================
# PART 4 — PROFESSIONAL COLOR THEME
# ================================================================

NAVY = "#17324F"
DARK_NAVY = "#10263D"
TEAL = "#087E83"

WHITE = "#FFFFFF"
LIGHT_BLUE = "#F1F6FA"
LIGHT_GRAY = "#F8FAFC"

TEXT = "#243447"
MUTED = "#64748B"
BORDER = "#DCE5ED"

# Image dimensions
WIDTH = 2000
MARGIN = 65

CONTENT_WIDTH = WIDTH - 2 * MARGIN


# ================================================================
# PART 5 — AUTOMATIC FONT DETECTION
# THIS FIXES THE PREVIOUS FONT ERROR
# ================================================================

regular_font_path = font_manager.findfont(
    font_manager.FontProperties(
        family="DejaVu Sans",
        weight="normal"
    )
)

bold_font_path = font_manager.findfont(
    font_manager.FontProperties(
        family="DejaVu Sans",
        weight="bold"
    )
)


def get_font(size, bold=False):

    font_path = (
        bold_font_path if bold else regular_font_path
    )

    return ImageFont.truetype(
        font_path,
        size
    )


title_font = get_font(51, bold=True)
subtitle_font = get_font(27)

header_font = get_font(30, bold=True)
body_font = get_font(31)
body_bold_font = get_font(31, bold=True)

note_font = get_font(24)

print("Fonts loaded successfully!")
print("Regular font:", regular_font_path)
print("Bold font:", bold_font_path)


# ================================================================
# PART 6 — TABLE HELPER FUNCTIONS
# ================================================================

def text_width(draw, text, font):

    box = draw.textbbox(
        (0, 0),
        str(text),
        font=font
    )

    return box[2] - box[0]


def wrap_text(draw, text, font, max_width):
    """
    Wrap text so that it fits inside the available width.
    """

    text = str(text)

    words = text.split()

    if not words:
        return [""]

    lines = []
    current = ""

    for word in words:

        trial = (
            current + " " + word
        ).strip()

        if text_width(draw, trial, font) <= max_width:

            current = trial

        else:

            if current:
                lines.append(current)
                current = word

            else:
                current = word

            # Handle unusually long words
            if text_width(draw, current, font) > max_width:

                letters = ""
                for character in current:

                    trial_letters = letters + character

                    if (
                        text_width(
                            draw,
                            trial_letters,
                            font
                        ) <= max_width
                    ):
                        letters = trial_letters

                    else:
                        if letters:
                            lines.append(letters)
                        letters = character

                current = letters

    if current:
        lines.append(current)

    return lines


def draw_cell_lines(
    draw,
    lines,
    x_left,
    x_right,
    y_top,
    row_height,
    font,
    color,
    align="left"
):
    """
    Draw a cell's text with vertical centering.
    """

    line_spacing = 43

    total_text_height = len(lines) * line_spacing

    start_y = (
        y_top
        + (row_height - total_text_height) / 2
    )

    for i, line in enumerate(lines):

        line_y = start_y + i * line_spacing

        if align == "left":

            draw.text(
                (x_left + 17, line_y),
                line,
                font=font,
                fill=color
            )

        else:

            width = text_width(
                draw,
                line,
                font
            )

            draw.text(
                (
                    x_right - 17 - width,
                    line_y
                ),
                line,
                font=font,
                fill=color
            )


# ================================================================
# PART 7 — MAIN PROFESSIONAL TABLE GENERATOR
# ================================================================

def create_styled_table(
    filename,
    title,
    subtitle,
    headers,
    rows,
    column_widths,
    notes=None
):

    if notes is None:
        notes = []

    # Temporary drawing surface for measuring text
    temp_image = Image.new(
        "RGB",
        (WIDTH, 100),
        WHITE
    )

    temp_draw = ImageDraw.Draw(temp_image)

    # Calculate column widths
    widths = [
        int(CONTENT_WIDTH * ratio)
        for ratio in column_widths
    ]

    widths[-1] = (
        CONTENT_WIDTH - sum(widths[:-1])
    )

    x_positions = [MARGIN]

    for width in widths:

        x_positions.append(
            x_positions[-1] + width
        )

    # Prepare text and row heights
    prepared_rows = []
    row_heights = []

    for row in rows:

        prepared_cells = []
        max_lines = 1

        for j, value in enumerate(row):

            font = (
                body_bold_font
                if j == 0
                else body_font
            )

            available_width = widths[j] - 40

            cell_lines = wrap_text(
                temp_draw,
                value,
                font,
                available_width
            )

            prepared_cells.append(cell_lines)

            max_lines = max(
                max_lines,
                len(cell_lines)
            )

        prepared_rows.append(prepared_cells)

        row_height = max(
            77,
            max_lines * 43 + 27
        )

        row_heights.append(row_height)

    # Prepare notes
    wrapped_notes = []

    for note in notes:

        wrapped_notes.extend(
            wrap_text(
                temp_draw,
                note,
                note_font,
                CONTENT_WIDTH
            )
        )

    # Layout measurements
    header_y = 230
    header_height = 83

    total_rows_height = sum(row_heights)
    notes_height = len(wrapped_notes) * 37

    HEIGHT = (
        header_y
        + header_height
        + total_rows_height
        + notes_height
        + 110
    )

    # Create final image
    image = Image.new(
        "RGB",
        (WIDTH, HEIGHT),
        WHITE
    )

    draw = ImageDraw.Draw(image)

    # Top accent
    draw.rectangle(
        [0, 0, WIDTH, 14],
        fill=TEAL
    )

    # Title
    draw.text(
        (MARGIN, 55),
        title,
        font=title_font,
        fill=NAVY
    )

    # Subtitle
    draw.text(
        (MARGIN, 137),
        subtitle,
        font=subtitle_font,
        fill=MUTED
    )

    # Divider above table
    draw.line(
        [
            MARGIN,
            206,
            WIDTH - MARGIN,
            206
        ],
        fill=BORDER,
        width=3
    )

    # Header background
    draw.rounded_rectangle(
        [
            MARGIN,
            header_y,
            WIDTH - MARGIN,
            header_y + header_height
        ],
        radius=10,
        fill=DARK_NAVY
    )

    # Draw headers
    for j, header in enumerate(headers):

        x0 = x_positions[j]
        x1 = x_positions[j + 1]

        if j == 0:

            draw.text(
                (
                    x0 + 17,
                    header_y + header_height / 2
                ),
                str(header),
                font=header_font,
                fill=WHITE,
                anchor="lm"
            )

        else:

            width = text_width(
                draw,
                header,
                header_font
            )

            draw.text(
                (
                    x1 - 17 - width,
                    header_y + 22
                ),
                str(header),
                font=header_font,
                fill=WHITE
            )

    # Draw rows
    current_y = header_y + header_height

    for i, row in enumerate(rows):

        height = row_heights[i]

        y0 = current_y
        y1 = current_y + height

        # Alternating backgrounds
        background = (
            WHITE if i % 2 == 0
            else LIGHT_BLUE
        )

        draw.rectangle(
            [
                MARGIN,
                y0,
                WIDTH - MARGIN,
                y1
            ],
            fill=background
        )

        # Row divider
        draw.line(
            [
                MARGIN,
                y1,
                WIDTH - MARGIN,
                y1
            ],
            fill=BORDER,
            width=2
        )

        # Cell text
        for j, value in enumerate(row):

            lines = prepared_rows[i][j]

            font = (
                body_bold_font
                if j == 0
                else body_font
            )

            color = TEXT

            # Highlight significant p-values
            if headers[j].startswith("p ("):

                try:

                    p_text = str(value)

                    if p_text.startswith("<"):
                        p_value = 0.0005
                    else:
                        p_value = float(p_text)

                    if p_value < 0.05:

                        color = TEAL
                        font = body_bold_font

                except (ValueError, TypeError):
                    pass

            draw_cell_lines(
                draw=draw,
                lines=lines,
                x_left=x_positions[j],
                x_right=x_positions[j + 1],
                y_top=y0,
                row_height=height,
                font=font,
                color=color,
                align=(
                    "left" if j == 0
                    else "right"
                )
            )

        current_y = y1

    # Draw notes
    note_y = current_y + 27

    for line in wrapped_notes:

        draw.text(
            (MARGIN, note_y),
            line,
            font=note_font,
            fill=MUTED
        )

        note_y += 37

    # Bottom accent
    draw.rectangle(
        [
            0,
            HEIGHT - 10,
            WIDTH,
            HEIGHT
        ],
        fill=TEAL
    )

    # Save PNG
    png_path = os.path.join(
        output_dir,
        filename + ".png"
    )

    image.save(
        png_path,
        dpi=(300, 300),
        optimize=True
    )

    # Save CSV
    csv_path = os.path.join(
        output_dir,
        filename + ".csv"
    )

    pd.DataFrame(
        rows,
        columns=headers
    ).to_csv(
        csv_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("Created:", filename)

    return png_path


# ================================================================
# PART 8 — TABLE 1: DATASET OVERVIEW
# ================================================================

total_original = len(df)
total_retained = len(df_work)

excluded = total_original - total_retained

mean_words = df_work["Word_Count"].mean()

median_percent = (
    df_work["Percent_Raised"].median()
)

successful = (
    df_work["Percent_Raised"] >= 100
).sum()

successful_percent = (
    successful / total_retained * 100
)

nominal_goals = (
    df_work["Goal$"] <= 2
).sum()


table1_rows = [

    [
        "Original fundraising campaigns",
        f"{total_original:,}"
    ],

    [
        "Campaigns retained after cleaning",
        f"{total_retained:,}"
    ],

    [
        "Campaigns excluded",
        f"{excluded:,}"
    ],

    [
        "Average description length",
        f"{mean_words:.2f} words"
    ],

    [
        "Median funding goal achieved",
        f"{median_percent:.2f}%"
    ],

    [
        "Campaigns reaching their goal",
        f"{successful:,} ({successful_percent:.1f}%)"
    ],

    [
        "Campaigns with goals of $2 or less",
        f"{nominal_goals:,}"
    ],

    [
        "Model A observations",
        f"{int(model_A.nobs):,}"
    ],

    [
        "Model B observations",
        f"{int(model_B.nobs):,}"
    ]

]


create_styled_table(

    filename="Table_1_Dataset_Overview",

    title="Table 1. Dataset Overview",

    subtitle="GoFundMe campaign data and preprocessing summary",

    headers=[
        "Characteristic",
        "Value"
    ],

    rows=table1_rows,

    column_widths=[
        0.72,
        0.28
    ],

    notes=[
        "Note: Records missing essential information were excluded. "
        "Fundraising durations reported in months were converted using 30 days per month.",

        "Model A excludes campaigns with nominal fundraising goals of $2 or less. "
        "Many campaign descriptions appear to be truncated."
    ]

)


# ================================================================
# PART 9 — TABLE 2: SEANCE EMOTIONAL CONSTRUCTS
# ================================================================

selected_emotions = [

    "Feelinglove_GALC",
    "Gratitude_GALC",
    "Hope_GALC",
    "Anger_GALC"

]


emotion_labels = {

    "Feelinglove_GALC": "Feeling Love",
    "Gratitude_GALC": "Gratitude",
    "Hope_GALC": "Hope",
    "Anger_GALC": "Anger"

}


table2_rows = []

for variable in selected_emotions:

    values = pd.to_numeric(
        df_merged[variable],
        errors="coerce"
    )

    nonzero = (values > 0).sum()

    percentage = (
        nonzero / len(df_merged) * 100
    )

    mean_score = values.mean()

    sd_score = values.std()

    table2_rows.append([

        emotion_labels[variable],

        f"{nonzero:,}",

        f"{percentage:.2f}%",

        f"{mean_score:.4f}",

        f"{sd_score:.4f}"

    ])


create_styled_table(

    filename="Table_2_SEANCE_Emotions",

    title="Table 2. SEANCE Emotional Constructs",

    subtitle="Descriptive statistics of the four selected GALC measures",

    headers=[
        "Emotion",
        "Nonzero n",
        "Nonzero %",
        "Mean",
        "SD"
    ],

    rows=table2_rows,

    column_widths=[
        0.32,
        0.17,
        0.17,
        0.17,
        0.17
    ],

    notes=[
        "Note: N = 1,231 campaigns. Nonzero n represents campaigns "
        "with a positive score for the selected emotional category.",

        "Mean and SD represent dictionary-based emotional scores. "
        "A zero score indicates no detected dictionary match."
    ]

)


# ================================================================
# PART 10 — REGRESSION FORMATTING FUNCTIONS
# ================================================================

variable_names = {

    "const": "Intercept",

    "Feelinglove_GALC": "Feeling Love",

    "Gratitude_GALC": "Gratitude",

    "Hope_GALC": "Hope",

    "Anger_GALC": "Anger",

    "GFM_hearts": "GFM Hearts",

    "FB_Shares": "Facebook Shares",

    "Length_Days": "Duration (days)"

}


def format_number(value):

    value = float(value)

    if abs(value) < 1:

        return f"{value:.4f}"

    else:

        return f"{value:,.2f}"


def format_pvalue(p):

    if p < 0.001:

        return "<0.001"

    return f"{p:.3f}"


def format_coefficient_se(model, variable):

    coefficient = model.params[variable]

    robust_se = model.bse[variable]

    return (
        f"{format_number(coefficient)} "
        f"({format_number(robust_se)})"
    )


# ================================================================
# PART 11 — TABLE 3: MAIN REGRESSION RESULTS
# ================================================================

table3_rows = []

for variable, label in variable_names.items():

    table3_rows.append([

        label,

        format_coefficient_se(
            model_A,
            variable
        ),

        format_pvalue(
            model_A.pvalues[variable]
        ),

        format_coefficient_se(
            model_B,
            variable
        ),

        format_pvalue(
            model_B.pvalues[variable]
        )

    ])


# Add model performance statistics
table3_rows.extend([

    [
        "Observations",
        f"{int(model_A.nobs):,}",
        "—",
        f"{int(model_B.nobs):,}",
        "—"
    ],

    [
        "R-squared",
        f"{model_A.rsquared:.4f}",
        "—",
        f"{model_B.rsquared:.4f}",
        "—"
    ],

    [
        "Adjusted R-squared",
        f"{model_A.rsquared_adj:.4f}",
        "—",
        f"{model_B.rsquared_adj:.4f}",
        "—"
    ]

])


create_styled_table(

    filename="Table_3_Main_Regression",

    title="Table 3. Main Regression Results",

    subtitle="Model A: Percent raised | Model B: Number of donors",

    headers=[
        "Predictor",
        "Model A b (SE)",
        "p (A)",
        "Model B b (SE)",
        "p (B)"
    ],

    rows=table3_rows,

    column_widths=[
        0.29,
        0.265,
        0.095,
        0.265,
        0.085
    ],

    notes=[
        "Note: Unstandardized OLS coefficients are reported with "
        "HC3 heteroskedasticity-robust standard errors in parentheses.",

        "Model A: N = 1,181. Model B: N = 1,192. "
        "Significant p-values (p < 0.05) are highlighted.",

        "SEANCE emotional variables are measured on their original "
        "proportion scale. Model B includes GoFundMe hearts."
    ]

)


# ================================================================
# PART 12 — TABLE 4: SENSITIVITY ANALYSIS
# ================================================================

def significant_findings(model):

    labels = {

        "Feelinglove_GALC": "Feeling Love",
        "Gratitude_GALC": "Gratitude",
        "Hope_GALC": "Hope",
        "Anger_GALC": "Anger",
        "GFM_hearts": "GFM Hearts",
        "FB_Shares": "FB Shares",
        "Length_Days": "Duration"

    }

    findings = []

    for variable in model.params.index:

        if variable == "const":
            continue

        if model.pvalues[variable] < 0.05:

            direction = (
                "positive"
                if model.params[variable] > 0
                else "negative"
            )

            p_value = format_pvalue(
                model.pvalues[variable]
            )

            if p_value.startswith("<"):

                p_text = "p" + p_value

            else:

                p_text = "p=" + p_value

            findings.append(
                f"{labels.get(variable, variable)} "
                f"({direction}; {p_text})"
            )

    if not findings:

        return "None at p < 0.05"

    return "; ".join(findings)


model_info = [

    (
        "Model A: Primary",
        "Percent raised",
        model_A
    ),

    (
        "Model A: Top 1% excluded",
        "Percent raised",
        model_A_sensitivity
    ),

    (
        "Model B: Primary",
        "Donor count",
        model_B
    ),

    (
        "Model B: Without hearts",
        "Donor count",
        model_B_no_hearts
    )

]


table4_rows = []

for label, outcome, model in model_info:

    table4_rows.append([

        label,

        outcome,

        f"{int(model.nobs):,}",

        f"{model.rsquared:.4f}",

        significant_findings(model)

    ])


create_styled_table(

    filename="Table_4_Sensitivity_Analysis",

    title="Table 4. Regression Sensitivity Analysis",

    subtitle="Comparison of primary and alternative specifications",

    headers=[
        "Specification",
        "Outcome",
        "N",
        "R2",
        "Significant finding"
    ],

    rows=table4_rows,

    column_widths=[
        0.275,
        0.19,
        0.09,
        0.10,
        0.345
    ],

    notes=[
        "Note: Model A sensitivity excludes funding percentages above "
        "the 99th percentile of the eligible Model A sample.",

        "Model B without hearts addresses the extremely high "
        "correlation between GoFundMe hearts and donor counts (r = 0.9986).",

        "None of the four selected emotional constructs was "
        "statistically significant at p < 0.05 in these specifications."
    ]

)


# ================================================================
# PART 13 — CREATE ONE ZIP FILE
# ================================================================

zip_path = (
    "/content/IS705_Homework3_Report_Tables.zip"
)

# Remove old ZIP if it exists
if os.path.exists(zip_path):

    os.remove(zip_path)


with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zip_file:

    for filename in sorted(
        os.listdir(output_dir)
    ):

        filepath = os.path.join(
            output_dir,
            filename
        )

        if os.path.isfile(filepath):

            zip_file.write(
                filepath,
                arcname=filename
            )


# ================================================================
# PART 14 — VERIFY EXPORTED FILES
# ================================================================

print("\n" + "=" * 70)

print(
    "ALL HOMEWORK 3 TABLES GENERATED SUCCESSFULLY!"
)

print("=" * 70)


print("\nGENERATED FILES:\n")

for filename in sorted(
    os.listdir(output_dir)
):

    filepath = os.path.join(
        output_dir,
        filename
    )

    file_size = os.path.getsize(
        filepath
    ) / 1024

    print(
        f"{filename} — {file_size:.1f} KB"
    )


# Verify PNG files
png_files = [

    filename

    for filename in os.listdir(output_dir)

    if filename.endswith(".png")

]

csv_files = [

    filename

    for filename in os.listdir(output_dir)

    if filename.endswith(".csv")

]


print("\nVERIFICATION:")

print(
    "PNG tables created:",
    len(png_files)
)

print(
    "CSV tables created:",
    len(csv_files)
)


assert len(png_files) == 4, \
    "Expected exactly four PNG tables."

assert len(csv_files) == 4, \
    "Expected exactly four CSV tables."


print("\nAll four tables are ready.")

print(
    "\nZIP file size:",
    round(
        os.path.getsize(zip_path) / (1024 * 1024),
        2
    ),
    "MB"
)


# ================================================================
# PART 15 — DOWNLOAD THE COMPLETE PACKAGE
# ================================================================

print("\nDownloading your report tables...")

files.download(zip_path)

print("\nDONE!")

## 8. Additional EmoLex analysis
I extended the original homework with the NRC EmoLex dictionary, using the same campaign descriptions and campaign IDs. I merged ten EmoLex scores into a **new copy** of the GALC dataset, keeping the original models intact. This analysis explores Trust, Anticipation, Sadness, and Fear as theoretically meaningful emotional themes.

In [ ]:
# ==========================================================
# HOMEWORK 3 - ADDITIONAL EMOLEX ANALYSIS
# STEP 2: IMPORT AND MERGE EMOLEX RESULTS
# ==========================================================

import pandas as pd
import numpy as np
import os
from google.colab import files


# ----------------------------------------------------------
# 1. Check that our original dataset is available
# ----------------------------------------------------------

if "df_merged" not in globals():
    raise RuntimeError(
        "Original df_merged dataset is not available. "
        "Please rerun your previous notebook cells."
    )


# ----------------------------------------------------------
# 2. Upload the new SEANCE output
# ----------------------------------------------------------

print("UPLOAD YOUR results_additional.csv FILE")

uploaded = files.upload()

filename = next(iter(uploaded))


# ----------------------------------------------------------
# 3. Read the new SEANCE results
# ----------------------------------------------------------

seance_additional = pd.read_csv(
    filename,
    encoding="utf-8-sig"
)

print("\nNEW SEANCE DATASET SHAPE:")
print(seance_additional.shape)


# ----------------------------------------------------------
# 4. Identify GALC and EmoLex columns
# ----------------------------------------------------------

galc_columns_new = [
    col for col in seance_additional.columns
    if col.endswith("_GALC")
]

emolex_columns = [
    col for col in seance_additional.columns
    if col.endswith("_EmoLex")
]

print("\nNUMBER OF GALC CONSTRUCTS:")
print(len(galc_columns_new))

print("\nNUMBER OF EMOLEX CONSTRUCTS:")
print(len(emolex_columns))

print("\nEMOLEX VARIABLES:")
print(emolex_columns)


# ----------------------------------------------------------
# 5. Extract Campaign_ID from filenames
# ----------------------------------------------------------

# Example: "300.txt" -> 300

seance_additional["Campaign_ID"] = (
    seance_additional["filename"]
    .astype(str)
    .str.strip()
    .str.replace("\\", "/", regex=False)
    .str.split("/")
    .str[-1]
    .str.extract(r"^(\d+)\.txt$", expand=False)
)

assert seance_additional["Campaign_ID"].notna().all(), \
    "Some filenames contain invalid campaign identifiers."

seance_additional["Campaign_ID"] = (
    seance_additional["Campaign_ID"].astype(int)
)


# ----------------------------------------------------------
# 6. Check for duplicate IDs
# ----------------------------------------------------------

print("\nDUPLICATE CAMPAIGN IDs:")

print(
    seance_additional["Campaign_ID"]
    .duplicated()
    .sum()
)


# ----------------------------------------------------------
# 7. Select only the new EmoLex variables
# ----------------------------------------------------------

# GALC already exists in our original dataset.
# We do not want duplicate GALC columns.

emolex_data = seance_additional[
    ["Campaign_ID"] + emolex_columns
].copy()


# ----------------------------------------------------------
# 8. Merge EmoLex scores with our original dataset
# ----------------------------------------------------------

df_with_emolex = df_merged.merge(
    emolex_data,
    on="Campaign_ID",
    how="left",
    validate="one_to_one",
    indicator="emolex_merge_status"
)


# ----------------------------------------------------------
# 9. Verify the merge
# ----------------------------------------------------------

print("\nMERGE VERIFICATION:")

print(
    "Original campaigns:",
    len(df_merged)
)

print(
    "New SEANCE records:",
    len(seance_additional)
)

print(
    "Combined dataset:",
    len(df_with_emolex)
)

print("\nMERGE STATUS:")

print(
    df_with_emolex[
        "emolex_merge_status"
    ].value_counts()
)


# ----------------------------------------------------------
# 10. Check for missing EmoLex scores
# ----------------------------------------------------------

print("\nMISSING EMOLEX VALUES:")

print(
    df_with_emolex[
        emolex_columns
    ].isna().sum()
)


# ----------------------------------------------------------
# 11. Display the new combined dataset
# ----------------------------------------------------------

print("\nFIRST 10 CAMPAIGNS WITH EMOLEX SCORES:")

display(
    df_with_emolex[
        [
            "Campaign_ID",
            "Percent_Raised",
            "Number_of_Donators",
            "Trust_EmoLex",
            "Anticipation_EmoLex",
            "Joy_EmoLex",
            "Sadness_EmoLex",
            "Fear_EmoLex"
        ]
    ].head(10)
)


# ----------------------------------------------------------
# 12. Final verification
# ----------------------------------------------------------

assert len(df_with_emolex) == len(df_merged), \
    "The number of campaigns changed."

assert (
    df_with_emolex["emolex_merge_status"] == "both"
).all(), "Some campaigns did not match."

assert (
    df_with_emolex[emolex_columns].notna().all().all()
), "Some EmoLex scores are missing."


print("\n" + "=" * 60)

print("SUCCESS! EMOLEX SCORES MERGED CORRECTLY.")

print("Original GALC analysis remains unchanged.")

print("=" * 60)

### Examine EmoLex coverage and correlations
Before fitting the additional models, I checked how often each emotion appeared and whether pairs of candidate emotions were strongly correlated. More dictionary matches do not automatically imply better measurement.

In [ ]:
# ==========================================================
# HOMEWORK 3 - ADDITIONAL EMOLEX ANALYSIS
# STEP 3: EXAMINE ALL EMOLEX CONSTRUCTS
# ==========================================================

import pandas as pd
import numpy as np

# ----------------------------------------------------------
# 1. Identify all EmoLex variables
# ----------------------------------------------------------

emolex_columns = [
    col for col in df_with_emolex.columns
    if col.endswith("_EmoLex")
]

print("TOTAL EMOLEX CONSTRUCTS:", len(emolex_columns))

print("\nEMOLEX VARIABLES:")
print(emolex_columns)


# ----------------------------------------------------------
# 2. Convert EmoLex scores to numerical values
# ----------------------------------------------------------

emolex_data = df_with_emolex[
    emolex_columns
].apply(
    pd.to_numeric,
    errors="coerce"
)


# ----------------------------------------------------------
# 3. Calculate descriptive statistics
# ----------------------------------------------------------

emolex_summary = pd.DataFrame({

    "Nonzero_Campaigns":
        (emolex_data > 0).sum(),

    "Nonzero_Percent":
        (emolex_data > 0).mean() * 100,

    "Mean":
        emolex_data.mean(),

    "Std_Dev":
        emolex_data.std(),

    "Minimum":
        emolex_data.min(),

    "Maximum":
        emolex_data.max(),

    "Unique_Values":
        emolex_data.nunique()

})


# ----------------------------------------------------------
# 4. Sort by number of campaigns with positive scores
# ----------------------------------------------------------

emolex_summary = emolex_summary.sort_values(
    "Nonzero_Campaigns",
    ascending=False
)

print("\nDISTRIBUTION OF ALL EMOLEX CONSTRUCTS:")

display(
    emolex_summary.round(4)
)


# ----------------------------------------------------------
# 5. Check constructs with zero variation
# ----------------------------------------------------------

print("\nCONSTRUCTS WITH ZERO VARIATION:")

zero_variation = emolex_summary[
    emolex_summary["Std_Dev"] == 0
]

if len(zero_variation) == 0:
    print("None. All EmoLex constructs have variation.")
else:
    print(zero_variation.index.tolist())


# ----------------------------------------------------------
# 6. Examine four theoretically relevant candidates
# ----------------------------------------------------------

candidate_emotions = [
    "Trust_EmoLex",
    "Anticipation_EmoLex",
    "Sadness_EmoLex",
    "Fear_EmoLex"
]

print("\nFOUR CANDIDATE EMOTIONAL CONSTRUCTS:")

display(
    emolex_summary.loc[
        candidate_emotions
    ].round(4)
)


# ----------------------------------------------------------
# 7. Check correlation among all EmoLex constructs
# ----------------------------------------------------------

print("\nCORRELATION MATRIX - ALL EMOLEX CONSTRUCTS:")

emolex_correlation = emolex_data.corr()

display(
    emolex_correlation.round(3)
)


# ----------------------------------------------------------
# 8. Check correlations among our four candidates
# ----------------------------------------------------------

print("\nCORRELATION MATRIX - FOUR CANDIDATES:")

candidate_correlation = (
    df_with_emolex[candidate_emotions]
    .corr()
)

display(
    candidate_correlation.round(3)
)


# ----------------------------------------------------------
# 9. Identify strongly correlated candidate variables
# ----------------------------------------------------------

print("\nSTRONG CORRELATION CHECK:")

strong_pairs = []

for i in range(len(candidate_emotions)):

    for j in range(i + 1, len(candidate_emotions)):

        var1 = candidate_emotions[i]
        var2 = candidate_emotions[j]

        correlation = candidate_correlation.loc[
            var1, var2
        ]

        if abs(correlation) >= 0.70:

            strong_pairs.append(
                (var1, var2, round(correlation, 3))
            )


if strong_pairs:

    print("Strong correlations detected:")

    for pair in strong_pairs:
        print(pair)

else:

    print(
        "No candidate emotion pairs have "
        "an absolute correlation >= 0.70."
    )


# ----------------------------------------------------------
# 10. Final verification
# ----------------------------------------------------------

print("\nFINAL VERIFICATION:")

print(
    "Total campaigns:",
    len(df_with_emolex)
)

print(
    "Total EmoLex variables:",
    len(emolex_columns)
)

print(
    "Missing EmoLex values:",
    emolex_data.isna().sum().sum()
)

print("\nSTEP 3 COMPLETED.")

### Prepare matching samples and VIF checks
To allow a fair comparison, I used the same campaigns for each matching GALC and EmoLex model, and checked multicollinearity before regression.

In [ ]:
# ==========================================================
# HOMEWORK 3 - ADDITIONAL EMOLEX ANALYSIS
# STEP 4: PREPARE REGRESSION DATA AND CHECK VIF
# ==========================================================

import pandas as pd
import numpy as np
import statsmodels.api as sm

from statsmodels.stats.outliers_influence import (
    variance_inflation_factor
)


# ----------------------------------------------------------
# 1. Select four EmoLex emotional constructs
# ----------------------------------------------------------

selected_emolex = [
    "Trust_EmoLex",
    "Anticipation_EmoLex",
    "Sadness_EmoLex",
    "Fear_EmoLex"
]

print("SELECTED EMOLEX CONSTRUCTS:")
print(selected_emolex)


# ----------------------------------------------------------
# 2. Define all independent variables
# ----------------------------------------------------------

emolex_predictors = selected_emolex + [
    "GFM_hearts",
    "FB_Shares",
    "Length_Days"
]

print("\nALL REGRESSION PREDICTORS:")
print(emolex_predictors)


# ----------------------------------------------------------
# 3. Prepare Model A dataset
# Dependent Variable: Percent_Raised
# Exclude nominal fundraising goals
# ----------------------------------------------------------

df_EmoLex_A = df_with_emolex.loc[
    ~df_with_emolex["Nominal_Goal"]
].dropna(
    subset=["Percent_Raised"] + emolex_predictors
).copy()


# ----------------------------------------------------------
# 4. Prepare Model B dataset
# Dependent Variable: Number_of_Donators
# Retain campaigns with nominal goals
# ----------------------------------------------------------

df_EmoLex_B = df_with_emolex.dropna(
    subset=["Number_of_Donators"] + emolex_predictors
).copy()


# ----------------------------------------------------------
# 5. Verify regression sample sizes
# ----------------------------------------------------------

print("\nREGRESSION SAMPLE SIZES:")

print(
    "EmoLex Model A:",
    len(df_EmoLex_A)
)

print(
    "EmoLex Model B:",
    len(df_EmoLex_B)
)


# ----------------------------------------------------------
# 6. Compare with original GALC model samples
# ----------------------------------------------------------

print("\nCOMPARISON WITH ORIGINAL GALC MODELS:")

print(
    "Original GALC Model A:",
    len(df_A_reg)
)

print(
    "New EmoLex Model A:",
    len(df_EmoLex_A)
)

print(
    "Original GALC Model B:",
    len(df_B_reg)
)

print(
    "New EmoLex Model B:",
    len(df_EmoLex_B)
)


# ----------------------------------------------------------
# 7. Verify that the same Campaign IDs are used
# ----------------------------------------------------------

same_A = (
    set(df_A_reg["Campaign_ID"])
    ==
    set(df_EmoLex_A["Campaign_ID"])
)

same_B = (
    set(df_B_reg["Campaign_ID"])
    ==
    set(df_EmoLex_B["Campaign_ID"])
)

print("\nSAMPLE MATCHING CHECK:")

print("Model A same campaigns:", same_A)
print("Model B same campaigns:", same_B)

assert same_A and same_B, (
    "The EmoLex and GALC regression samples differ."
)


# ----------------------------------------------------------
# 8. Define VIF calculation function
# ----------------------------------------------------------

def calculate_emolex_vif(data, variables):

    X = data[variables].astype(float)

    X = sm.add_constant(
        X,
        has_constant="add"
    )

    vif_table = pd.DataFrame({

        "Variable": variables,

        "VIF": [
            variance_inflation_factor(
                X.values,
                X.columns.get_loc(variable)
            )
            for variable in variables
        ]

    })

    return vif_table


# ----------------------------------------------------------
# 9. Calculate VIF for EmoLex Model A
# ----------------------------------------------------------

vif_EmoLex_A = calculate_emolex_vif(
    df_EmoLex_A,
    emolex_predictors
)

print("\nEMOLEX MODEL A - VIF RESULTS:")

display(
    vif_EmoLex_A.round(3)
)


# ----------------------------------------------------------
# 10. Calculate VIF for EmoLex Model B
# ----------------------------------------------------------

vif_EmoLex_B = calculate_emolex_vif(
    df_EmoLex_B,
    emolex_predictors
)

print("\nEMOLEX MODEL B - VIF RESULTS:")

display(
    vif_EmoLex_B.round(3)
)


# ----------------------------------------------------------
# 11. Identify potential multicollinearity
# ----------------------------------------------------------

print("\nMODEL A: VARIABLES WITH VIF >= 5")

print(
    vif_EmoLex_A.loc[
        vif_EmoLex_A["VIF"] >= 5,
        "Variable"
    ].tolist()
)


print("\nMODEL B: VARIABLES WITH VIF >= 5")

print(
    vif_EmoLex_B.loc[
        vif_EmoLex_B["VIF"] >= 5,
        "Variable"
    ].tolist()
)


# ----------------------------------------------------------
# 12. Final verification
# ----------------------------------------------------------

print("\nFINAL VERIFICATION:")

print(
    "Total predictors:",
    len(emolex_predictors)
)

print(
    "Model A observations:",
    len(df_EmoLex_A)
)

print(
    "Model B observations:",
    len(df_EmoLex_B)
)

print("\nSTEP 4 COMPLETED.")

## 9. EmoLex regression models
I used the same two outcomes and the same non-emotional predictors as in the original GALC analysis. I estimated OLS with HC3 robust standard errors.

In [ ]:
# ==========================================================
# HOMEWORK 3 - ADDITIONAL EMOLEX ANALYSIS
# STEP 5: REGRESSION MODEL A
# DEPENDENT VARIABLE: PERCENT_RAISED
# ==========================================================

import pandas as pd
import numpy as np
import statsmodels.api as sm


# ----------------------------------------------------------
# 1. Define EmoLex predictors
# ----------------------------------------------------------

emolex_predictors = [
    "Trust_EmoLex",
    "Anticipation_EmoLex",
    "Sadness_EmoLex",
    "Fear_EmoLex",
    "GFM_hearts",
    "FB_Shares",
    "Length_Days"
]


# ----------------------------------------------------------
# 2. Prepare independent variables
# ----------------------------------------------------------

X_EmoLex_A = df_EmoLex_A[
    emolex_predictors
].astype(float)


# ----------------------------------------------------------
# 3. Prepare dependent variable
# ----------------------------------------------------------

y_EmoLex_A = df_EmoLex_A[
    "Percent_Raised"
].astype(float)


# ----------------------------------------------------------
# 4. Add intercept
# ----------------------------------------------------------

X_EmoLex_A = sm.add_constant(
    X_EmoLex_A,
    has_constant="add"
)


# ----------------------------------------------------------
# 5. Estimate regression using HC3 robust SE
# ----------------------------------------------------------

model_EmoLex_A = sm.OLS(
    y_EmoLex_A,
    X_EmoLex_A
).fit(
    cov_type="HC3"
)


# ----------------------------------------------------------
# 6. Display complete regression summary
# ----------------------------------------------------------

print("=" * 70)
print("EMOLEX MODEL A: PREDICTING FUNDRAISING SUCCESS")
print("=" * 70)

print(model_EmoLex_A.summary())


# ----------------------------------------------------------
# 7. Create regression results table
# ----------------------------------------------------------

results_EmoLex_A = pd.DataFrame({

    "Variable":
        model_EmoLex_A.params.index,

    "Coefficient":
        model_EmoLex_A.params.values,

    "Robust_SE":
        model_EmoLex_A.bse.values,

    "P_Value":
        model_EmoLex_A.pvalues.values,

    "CI_Lower":
        model_EmoLex_A.conf_int().iloc[:, 0].values,

    "CI_Upper":
        model_EmoLex_A.conf_int().iloc[:, 1].values

})


print("\nEMOLEX MODEL A: REGRESSION RESULTS TABLE")

display(
    results_EmoLex_A.round(5)
)


# ----------------------------------------------------------
# 8. Display model performance
# ----------------------------------------------------------

print("\nEMOLEX MODEL A PERFORMANCE:")

print(
    "Observations:",
    int(model_EmoLex_A.nobs)
)

print(
    "R-squared:",
    round(model_EmoLex_A.rsquared, 4)
)

print(
    "Adjusted R-squared:",
    round(model_EmoLex_A.rsquared_adj, 4)
)

print(
    "Overall model p-value:",
    model_EmoLex_A.f_pvalue
)


# ----------------------------------------------------------
# 9. Identify significant predictors
# ----------------------------------------------------------

print("\nSTATISTICALLY SIGNIFICANT PREDICTORS (p < 0.05):")

significant_EmoLex_A = results_EmoLex_A[
    (results_EmoLex_A["P_Value"] < 0.05)
    &
    (results_EmoLex_A["Variable"] != "const")
]

if len(significant_EmoLex_A) > 0:

    display(
        significant_EmoLex_A.round(5)
    )

else:

    print(
        "No statistically significant predictors."
    )


# ----------------------------------------------------------
# 10. Compare GALC and EmoLex Model A
# ----------------------------------------------------------

comparison_Model_A = pd.DataFrame({

    "Statistic": [
        "Observations",
        "R-squared",
        "Adjusted R-squared",
        "Overall model p-value"
    ],

    "GALC_Model_A": [
        int(model_A.nobs),
        model_A.rsquared,
        model_A.rsquared_adj,
        model_A.f_pvalue
    ],

    "EmoLex_Model_A": [
        int(model_EmoLex_A.nobs),
        model_EmoLex_A.rsquared,
        model_EmoLex_A.rsquared_adj,
        model_EmoLex_A.f_pvalue
    ]

})


print("\nGALC VS EMOLEX: MODEL A COMPARISON")

display(
    comparison_Model_A.round(5)
)


print("\nSTEP 5 COMPLETED SUCCESSFULLY.")

In [ ]:
# ==========================================================
# HOMEWORK 3 - ADDITIONAL EMOLEX ANALYSIS
# STEP 6: REGRESSION MODEL B
# DEPENDENT VARIABLE: NUMBER OF DONORS
# ==========================================================

import pandas as pd
import numpy as np
import statsmodels.api as sm


# ----------------------------------------------------------
# 1. Define EmoLex independent variables
# ----------------------------------------------------------

emolex_predictors = [
    "Trust_EmoLex",
    "Anticipation_EmoLex",
    "Sadness_EmoLex",
    "Fear_EmoLex",
    "GFM_hearts",
    "FB_Shares",
    "Length_Days"
]


# ----------------------------------------------------------
# 2. Prepare dependent and independent variables
# ----------------------------------------------------------

X_EmoLex_B = df_EmoLex_B[
    emolex_predictors
].astype(float)

y_EmoLex_B = df_EmoLex_B[
    "Number_of_Donators"
].astype(float)


# ----------------------------------------------------------
# 3. Add intercept
# ----------------------------------------------------------

X_EmoLex_B = sm.add_constant(
    X_EmoLex_B,
    has_constant="add"
)


# ----------------------------------------------------------
# 4. Estimate OLS with HC3 robust standard errors
# ----------------------------------------------------------

model_EmoLex_B = sm.OLS(
    y_EmoLex_B,
    X_EmoLex_B
).fit(cov_type="HC3")


# ----------------------------------------------------------
# 5. Display complete regression summary
# ----------------------------------------------------------

print("=" * 70)
print("EMOLEX MODEL B: PREDICTING NUMBER OF DONORS")
print("=" * 70)

print(model_EmoLex_B.summary())


# ----------------------------------------------------------
# 6. Create regression results table
# ----------------------------------------------------------

results_EmoLex_B = pd.DataFrame({

    "Variable":
        model_EmoLex_B.params.index,

    "Coefficient":
        model_EmoLex_B.params.values,

    "Robust_SE":
        model_EmoLex_B.bse.values,

    "P_Value":
        model_EmoLex_B.pvalues.values,

    "CI_Lower":
        model_EmoLex_B.conf_int().iloc[:, 0].values,

    "CI_Upper":
        model_EmoLex_B.conf_int().iloc[:, 1].values

})


print("\nEMOLEX MODEL B: REGRESSION RESULTS")

display(
    results_EmoLex_B.round(5)
)


# ----------------------------------------------------------
# 7. Display model performance
# ----------------------------------------------------------

print("\nEMOLEX MODEL B PERFORMANCE:")

print(
    "Observations:",
    int(model_EmoLex_B.nobs)
)

print(
    "R-squared:",
    round(model_EmoLex_B.rsquared, 4)
)

print(
    "Adjusted R-squared:",
    round(model_EmoLex_B.rsquared_adj, 4)
)

print(
    "Overall model p-value:",
    model_EmoLex_B.f_pvalue
)


# ----------------------------------------------------------
# 8. Identify statistically significant predictors
# ----------------------------------------------------------

print("\nSIGNIFICANT PREDICTORS (p < 0.05):")

significant_EmoLex_B = results_EmoLex_B[
    (results_EmoLex_B["P_Value"] < 0.05)
    &
    (results_EmoLex_B["Variable"] != "const")
]

if len(significant_EmoLex_B) > 0:

    display(
        significant_EmoLex_B.round(5)
    )

else:

    print("No statistically significant predictors.")


# ----------------------------------------------------------
# 9. Compare GALC and EmoLex Model B
# ----------------------------------------------------------

comparison_Model_B = pd.DataFrame({

    "Statistic": [
        "Observations",
        "R-squared",
        "Adjusted R-squared",
        "Overall model p-value"
    ],

    "GALC_Model_B": [
        int(model_B.nobs),
        model_B.rsquared,
        model_B.rsquared_adj,
        model_B.f_pvalue
    ],

    "EmoLex_Model_B": [
        int(model_EmoLex_B.nobs),
        model_EmoLex_B.rsquared,
        model_EmoLex_B.rsquared_adj,
        model_EmoLex_B.f_pvalue
    ]

})


print("\nGALC VS EMOLEX: MODEL B COMPARISON")

display(
    comparison_Model_B.round(5)
)


# ----------------------------------------------------------
# 10. Final verification
# ----------------------------------------------------------

assert len(df_EmoLex_B) == len(df_B_reg), \
    "Model B sample size does not match original GALC."

print("\nSTEP 6 COMPLETED SUCCESSFULLY.")

## 10. EmoLex sensitivity analyses
I repeated Model A after removing the top 1% of funding percentages and repeated Model B without the GoFundMe hearts predictor. These checks examine whether extreme outcomes and nearly duplicated engagement measures influence the findings.

In [ ]:
# ==========================================================
# HOMEWORK 3 - ADDITIONAL EMOLEX ANALYSIS
# STEP 7: MODEL A SENSITIVITY ANALYSIS
# ==========================================================

import pandas as pd
import numpy as np
import statsmodels.api as sm


# ----------------------------------------------------------
# 1. Calculate the same 99th percentile cutoff
# used in our original GALC analysis
# ----------------------------------------------------------

cutoff_99 = df_A_reg["Percent_Raised"].quantile(0.99)

print("99TH PERCENTILE CUTOFF:")
print(round(cutoff_99, 2))


# ----------------------------------------------------------
# 2. Create EmoLex sensitivity dataset
# ----------------------------------------------------------

df_EmoLex_A_sensitivity = df_EmoLex_A.loc[
    df_EmoLex_A["Percent_Raised"] <= cutoff_99
].copy()

print("\nSAMPLE SIZE VERIFICATION:")

print("Original EmoLex Model A:", len(df_EmoLex_A))

print(
    "Sensitivity EmoLex Model A:",
    len(df_EmoLex_A_sensitivity)
)

print(
    "Campaigns excluded:",
    len(df_EmoLex_A) - len(df_EmoLex_A_sensitivity)
)


# ----------------------------------------------------------
# 3. Verify same campaigns as GALC sensitivity model
# ----------------------------------------------------------

same_campaigns = (
    set(df_EmoLex_A_sensitivity["Campaign_ID"])
    ==
    set(df_A_sensitivity["Campaign_ID"])
)

print("\nSAME CAMPAIGNS AS GALC SENSITIVITY MODEL:")
print(same_campaigns)

assert same_campaigns, (
    "EmoLex and GALC sensitivity samples do not match."
)


# ----------------------------------------------------------
# 4. Define independent variables
# ----------------------------------------------------------

emolex_predictors = [
    "Trust_EmoLex",
    "Anticipation_EmoLex",
    "Sadness_EmoLex",
    "Fear_EmoLex",
    "GFM_hearts",
    "FB_Shares",
    "Length_Days"
]


# ----------------------------------------------------------
# 5. Prepare regression variables
# ----------------------------------------------------------

X_EmoLex_A_sensitivity = sm.add_constant(
    df_EmoLex_A_sensitivity[
        emolex_predictors
    ].astype(float),
    has_constant="add"
)

y_EmoLex_A_sensitivity = (
    df_EmoLex_A_sensitivity[
        "Percent_Raised"
    ].astype(float)
)


# ----------------------------------------------------------
# 6. Estimate OLS with HC3 robust standard errors
# ----------------------------------------------------------

model_EmoLex_A_sensitivity = sm.OLS(
    y_EmoLex_A_sensitivity,
    X_EmoLex_A_sensitivity
).fit(cov_type="HC3")


# ----------------------------------------------------------
# 7. Display regression summary
# ----------------------------------------------------------

print("\n" + "=" * 70)
print("EMOLEX MODEL A: SENSITIVITY REGRESSION")
print("=" * 70)

print(model_EmoLex_A_sensitivity.summary())


# ----------------------------------------------------------
# 8. Compare original and sensitivity coefficients
# ----------------------------------------------------------

comparison_EmoLex_A = pd.DataFrame({

    "Variable":
        model_EmoLex_A.params.index,

    "Original_Coefficient":
        model_EmoLex_A.params.values,

    "Original_SE":
        model_EmoLex_A.bse.values,

    "Original_P":
        model_EmoLex_A.pvalues.values,

    "Sensitivity_Coefficient":
        model_EmoLex_A_sensitivity.params.values,

    "Sensitivity_SE":
        model_EmoLex_A_sensitivity.bse.values,

    "Sensitivity_P":
        model_EmoLex_A_sensitivity.pvalues.values

})

print("\nEMOLEX MODEL A: REGRESSION COMPARISON")

display(comparison_EmoLex_A.round(4))


# ----------------------------------------------------------
# 9. Compare model performance
# ----------------------------------------------------------

performance_EmoLex_A = pd.DataFrame({

    "Statistic": [
        "Observations",
        "R-squared",
        "Adjusted R-squared",
        "Overall model p-value"
    ],

    "Original_EmoLex": [
        model_EmoLex_A.nobs,
        model_EmoLex_A.rsquared,
        model_EmoLex_A.rsquared_adj,
        model_EmoLex_A.f_pvalue
    ],

    "Sensitivity_EmoLex": [
        model_EmoLex_A_sensitivity.nobs,
        model_EmoLex_A_sensitivity.rsquared,
        model_EmoLex_A_sensitivity.rsquared_adj,
        model_EmoLex_A_sensitivity.f_pvalue
    ]

})

print("\nMODEL PERFORMANCE COMPARISON:")

display(performance_EmoLex_A.round(5))


# ----------------------------------------------------------
# 10. Compare GALC and EmoLex sensitivity models
# ----------------------------------------------------------

dictionary_comparison = pd.DataFrame({

    "Statistic": [
        "Observations",
        "R-squared",
        "Adjusted R-squared",
        "Overall model p-value"
    ],

    "GALC_Sensitivity": [
        model_A_sensitivity.nobs,
        model_A_sensitivity.rsquared,
        model_A_sensitivity.rsquared_adj,
        model_A_sensitivity.f_pvalue
    ],

    "EmoLex_Sensitivity": [
        model_EmoLex_A_sensitivity.nobs,
        model_EmoLex_A_sensitivity.rsquared,
        model_EmoLex_A_sensitivity.rsquared_adj,
        model_EmoLex_A_sensitivity.f_pvalue
    ]

})

print("\nGALC VS EMOLEX: SENSITIVITY COMPARISON")

display(dictionary_comparison.round(5))


# ----------------------------------------------------------
# 11. Identify statistically significant predictors
# ----------------------------------------------------------

print("\nSIGNIFICANT PREDICTORS IN EMOLEX SENSITIVITY MODEL:")

significant_emolex_sensitivity = comparison_EmoLex_A.loc[
    (comparison_EmoLex_A["Sensitivity_P"] < 0.05)
    &
    (comparison_EmoLex_A["Variable"] != "const")
]

if len(significant_emolex_sensitivity) > 0:

    display(significant_emolex_sensitivity.round(4))

else:

    print("No statistically significant predictors.")


print("\nSTEP 7 COMPLETED SUCCESSFULLY.")

In [ ]:
# ==========================================================
# HOMEWORK 3 - ADDITIONAL EMOLEX ANALYSIS
# STEP 8: MODEL B SENSITIVITY ANALYSIS
# REMOVE GOFUNDME HEARTS
# ==========================================================

import pandas as pd
import numpy as np
import statsmodels.api as sm


# ----------------------------------------------------------
# 1. Define predictors without GFM_hearts
# ----------------------------------------------------------

emolex_predictors_no_hearts = [
    "Trust_EmoLex",
    "Anticipation_EmoLex",
    "Sadness_EmoLex",
    "Fear_EmoLex",
    "FB_Shares",
    "Length_Days"
]


# ----------------------------------------------------------
# 2. Prepare independent variables
# ----------------------------------------------------------

X_EmoLex_B_no_hearts = df_EmoLex_B[
    emolex_predictors_no_hearts
].astype(float)


# ----------------------------------------------------------
# 3. Prepare dependent variable
# ----------------------------------------------------------

y_EmoLex_B_no_hearts = df_EmoLex_B[
    "Number_of_Donators"
].astype(float)


# ----------------------------------------------------------
# 4. Add intercept
# ----------------------------------------------------------

X_EmoLex_B_no_hearts = sm.add_constant(
    X_EmoLex_B_no_hearts,
    has_constant="add"
)


# ----------------------------------------------------------
# 5. Estimate OLS with HC3 robust standard errors
# ----------------------------------------------------------

model_EmoLex_B_no_hearts = sm.OLS(
    y_EmoLex_B_no_hearts,
    X_EmoLex_B_no_hearts
).fit(cov_type="HC3")


# ----------------------------------------------------------
# 6. Display complete regression summary
# ----------------------------------------------------------

print("=" * 70)
print("EMOLEX MODEL B: WITHOUT GOFUNDME HEARTS")
print("=" * 70)

print(model_EmoLex_B_no_hearts.summary())


# ----------------------------------------------------------
# 7. Create regression results table
# ----------------------------------------------------------

results_EmoLex_B_no_hearts = pd.DataFrame({

    "Variable":
        model_EmoLex_B_no_hearts.params.index,

    "Coefficient":
        model_EmoLex_B_no_hearts.params.values,

    "Robust_SE":
        model_EmoLex_B_no_hearts.bse.values,

    "P_Value":
        model_EmoLex_B_no_hearts.pvalues.values,

    "CI_Lower":
        model_EmoLex_B_no_hearts.conf_int().iloc[:, 0].values,

    "CI_Upper":
        model_EmoLex_B_no_hearts.conf_int().iloc[:, 1].values

})


print("\nEMOLEX MODEL B: RESULTS WITHOUT HEARTS")

display(
    results_EmoLex_B_no_hearts.round(5)
)


# ----------------------------------------------------------
# 8. Compare original EmoLex Model B
#    with sensitivity Model B
# ----------------------------------------------------------

comparison_EmoLex_B = pd.DataFrame({

    "Variable":
        model_EmoLex_B_no_hearts.params.index,

    "Original_Coefficient":
        model_EmoLex_B.params.reindex(
            model_EmoLex_B_no_hearts.params.index
        ).values,

    "Original_P":
        model_EmoLex_B.pvalues.reindex(
            model_EmoLex_B_no_hearts.params.index
        ).values,

    "Without_Hearts_Coefficient":
        model_EmoLex_B_no_hearts.params.values,

    "Without_Hearts_SE":
        model_EmoLex_B_no_hearts.bse.values,

    "Without_Hearts_P":
        model_EmoLex_B_no_hearts.pvalues.values

})


print("\nEMOLEX MODEL B: COEFFICIENT COMPARISON")

display(
    comparison_EmoLex_B.round(5)
)


# ----------------------------------------------------------
# 9. Compare model performance
# ----------------------------------------------------------

performance_EmoLex_B = pd.DataFrame({

    "Statistic": [
        "Observations",
        "R-squared",
        "Adjusted R-squared",
        "Overall model p-value"
    ],

    "Original_EmoLex": [
        model_EmoLex_B.nobs,
        model_EmoLex_B.rsquared,
        model_EmoLex_B.rsquared_adj,
        model_EmoLex_B.f_pvalue
    ],

    "Without_Hearts": [
        model_EmoLex_B_no_hearts.nobs,
        model_EmoLex_B_no_hearts.rsquared,
        model_EmoLex_B_no_hearts.rsquared_adj,
        model_EmoLex_B_no_hearts.f_pvalue
    ]

})


print("\nEMOLEX MODEL B: PERFORMANCE COMPARISON")

display(
    performance_EmoLex_B.round(5)
)


# ----------------------------------------------------------
# 10. Compare GALC and EmoLex without hearts
# ----------------------------------------------------------

comparison_GALC_EmoLex_B = pd.DataFrame({

    "Statistic": [
        "Observations",
        "R-squared",
        "Adjusted R-squared",
        "Overall model p-value"
    ],

    "GALC_Without_Hearts": [
        model_B_no_hearts.nobs,
        model_B_no_hearts.rsquared,
        model_B_no_hearts.rsquared_adj,
        model_B_no_hearts.f_pvalue
    ],

    "EmoLex_Without_Hearts": [
        model_EmoLex_B_no_hearts.nobs,
        model_EmoLex_B_no_hearts.rsquared,
        model_EmoLex_B_no_hearts.rsquared_adj,
        model_EmoLex_B_no_hearts.f_pvalue
    ]

})


print("\nGALC VS EMOLEX: MODEL B WITHOUT HEARTS")

display(
    comparison_GALC_EmoLex_B.round(5)
)


# ----------------------------------------------------------
# 11. Identify statistically significant predictors
# ----------------------------------------------------------

print("\nSIGNIFICANT PREDICTORS WITHOUT HEARTS (p < 0.05):")

significant_EmoLex_B_no_hearts = (
    results_EmoLex_B_no_hearts.loc[
        (results_EmoLex_B_no_hearts["P_Value"] < 0.05)
        &
        (results_EmoLex_B_no_hearts["Variable"] != "const")
    ]
)

if len(significant_EmoLex_B_no_hearts) > 0:

    display(
        significant_EmoLex_B_no_hearts.round(5)
    )

else:

    print("No statistically significant predictors.")


# ----------------------------------------------------------
# 12. Verify sample sizes
# ----------------------------------------------------------

print("\nSAMPLE VERIFICATION:")

print(
    "Original EmoLex Model B:",
    int(model_EmoLex_B.nobs)
)

print(
    "EmoLex without hearts:",
    int(model_EmoLex_B_no_hearts.nobs)
)

print(
    "Original GALC without hearts:",
    int(model_B_no_hearts.nobs)
)

assert (
    int(model_EmoLex_B_no_hearts.nobs)
    ==
    int(model_B_no_hearts.nobs)
), "Sample sizes do not match."

assert (
    set(df_EmoLex_B["Campaign_ID"])
    ==
    set(df_B_reg["Campaign_ID"])
), "Campaign IDs do not match."


print("\nSTEP 8 COMPLETED SUCCESSFULLY.")

## 11. Export comparison tables
The final cell summarizes coefficients, standard errors, p-values, sample sizes, R², and significant predictors for the GALC and EmoLex models. Only aggregate result tables should be considered for public sharing; review any outputs before committing them.

In [ ]:
# ==========================================================
# HOMEWORK 3 - ADDITIONAL EMOLEX ANALYSIS
# STEP 9: CREATE FINAL TABLES FOR THE REPORT
# ==========================================================

import pandas as pd
import zipfile
from google.colab import files


# ----------------------------------------------------------
# TABLE 5: EMOLEX REGRESSION RESULTS
# ----------------------------------------------------------

emotion_names = {
    "Trust_EmoLex": "Trust",
    "Anticipation_EmoLex": "Anticipation",
    "Sadness_EmoLex": "Sadness",
    "Fear_EmoLex": "Fear",
    "GFM_hearts": "GoFundMe Hearts",
    "FB_Shares": "Facebook Shares",
    "Length_Days": "Fundraising Duration"
}


def format_p(p):
    if p < 0.001:
        return "<0.001"
    return f"{p:.4f}"


table5_rows = []

for variable, label in emotion_names.items():

    table5_rows.append({

        "Predictor": label,

        "Model_A_Coefficient":
            model_EmoLex_A.params[variable],

        "Model_A_Robust_SE":
            model_EmoLex_A.bse[variable],

        "Model_A_P":
            format_p(model_EmoLex_A.pvalues[variable]),

        "Model_B_Coefficient":
            model_EmoLex_B.params[variable],

        "Model_B_Robust_SE":
            model_EmoLex_B.bse[variable],

        "Model_B_P":
            format_p(model_EmoLex_B.pvalues[variable])

    })


table5 = pd.DataFrame(table5_rows)

print("\nTABLE 5: EMOLEX REGRESSION RESULTS")

display(table5.round(4))


# ----------------------------------------------------------
# TABLE 6: GALC VS EMOLEX MODEL COMPARISON
# ----------------------------------------------------------

model_information = [

    ("GALC", "Percent Raised", "Primary",
     model_A),

    ("EmoLex", "Percent Raised", "Primary",
     model_EmoLex_A),

    ("GALC", "Number of Donors", "Primary",
     model_B),

    ("EmoLex", "Number of Donors", "Primary",
     model_EmoLex_B),

    ("GALC", "Percent Raised", "Exclude Top 1%",
     model_A_sensitivity),

    ("EmoLex", "Percent Raised", "Exclude Top 1%",
     model_EmoLex_A_sensitivity),

    ("GALC", "Number of Donors", "Without Hearts",
     model_B_no_hearts),

    ("EmoLex", "Number of Donors", "Without Hearts",
     model_EmoLex_B_no_hearts)

]


galc_emotions = [
    "Feelinglove_GALC",
    "Gratitude_GALC",
    "Hope_GALC",
    "Anger_GALC"
]

emolex_emotions = [
    "Trust_EmoLex",
    "Anticipation_EmoLex",
    "Sadness_EmoLex",
    "Fear_EmoLex"
]

other_predictors = [
    "GFM_hearts",
    "FB_Shares",
    "Length_Days"
]


table6_rows = []

for dictionary, outcome, specification, model in model_information:

    emotion_variables = (
        galc_emotions
        if dictionary == "GALC"
        else emolex_emotions
    )

    significant_emotions = [
        v for v in emotion_variables
        if v in model.pvalues.index
        and model.pvalues[v] < 0.05
    ]

    significant_other = [
        v for v in other_predictors
        if v in model.pvalues.index
        and model.pvalues[v] < 0.05
    ]

    table6_rows.append({

        "Dictionary": dictionary,

        "Outcome": outcome,

        "Specification": specification,

        "N": int(model.nobs),

        "R_Squared": model.rsquared,

        "Adjusted_R_Squared": model.rsquared_adj,

        "Significant_Emotions":
            ", ".join(significant_emotions)
            if significant_emotions else "None",

        "Other_Significant_Predictors":
            ", ".join(significant_other)
            if significant_other else "None"

    })


table6 = pd.DataFrame(table6_rows)

print("\nTABLE 6: GALC VS EMOLEX COMPARISON")

display(table6.round(4))


# ----------------------------------------------------------
# SAVE BOTH TABLES
# ----------------------------------------------------------

table5.to_csv(
    "Table_5_EmoLex_Regression.csv",
    index=False
)

table6.to_csv(
    "Table_6_GALC_EmoLex_Comparison.csv",
    index=False
)


# ----------------------------------------------------------
# CREATE ONE ZIP FILE FOR EASY DOWNLOAD
# ----------------------------------------------------------

zip_name = "Homework3_Additional_EmoLex_Tables.zip"

with zipfile.ZipFile(zip_name, "w") as z:

    z.write("Table_5_EmoLex_Regression.csv")

    z.write("Table_6_GALC_EmoLex_Comparison.csv")


print("\n" + "=" * 60)

print("STEP 9 COMPLETED SUCCESSFULLY.")

print("Both tables have been created.")

print("=" * 60)


files.download(zip_name)

## 12. Key findings and limitations

**Funding percentage (Model A).** Among 1,181 campaigns, the GALC model had R² = 0.0293 and the EmoLex model R² = 0.0247. No selected emotion was significant at p < .05. Fundraising duration was negatively associated with percentage raised in both models. After excluding the top 1% of fundraising percentages (N = 1,169), no selected emotional measure became significant.

**Donor count (Model B).** Among 1,192 campaigns, both full models had R² ≈ 0.9973, driven by near-duplication between GoFundMe hearts and donor counts (r ≈ 0.9986). Without hearts, R² declined to 0.5616 for GALC and 0.5642 for EmoLex. Facebook shares were positively associated with donor count in both reduced models; selected emotions were not significant.

**Interpretation.** These are *associations* rather than causal estimates. Emotion lexicons measure word matches, not readers' feelings. Campaign text was short (about 63 words on average), some descriptions appear truncated, funding outcomes are highly skewed, and the selected constructs are exploratory. The original source and sensitive campaign-level data are not supplied in this public notebook.